# Fake.br: Isolation Forest não supervisionado — hybrid_rank_q95

Notebook autossuficiente derivado do notebook de melhorias. Calcula a vista full-text, o ensemble windowed de três amostragens, agregadores secundários e a fusão por percentis 50/50. O teste histórico já foi inspecionado e permanece avaliação interna exploratória; anomalia não comprova falsidade. Esta construção não modifica notebooks protegidos.

## 2. Imports

Semente fixa de 42. As versões do ambiente ficam registradas para reprodução.

In [ ]:
from pathlib import Path
from urllib.request import urlopen
from zipfile import ZipFile
from itertools import combinations
from importlib.metadata import version
import hashlib
import platform

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display
from sklearn.ensemble import IsolationForest
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score, average_precision_score, precision_score, recall_score,
    f1_score, confusion_matrix, roc_curve, precision_recall_curve,
    ConfusionMatrixDisplay,
)

In [ ]:
RANDOM_STATE = 42
pd.set_option("display.max_rows", 80)
pd.set_option("display.max_columns", 20)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.2})
print("Python:", platform.python_version())
print({name: version(name) for name in ["numpy", "pandas", "scikit-learn", "matplotlib", "ipykernel"]})

## 3. Carregamento dos dados

Mesmo ZIP e revisão fixa do notebook original. Textos completos e metadados são
carregados para auditoria; o truncamento acontece ANTES da extração usada nos
modelos. Nenhuma notícia é excluída por comprimento.

In [ ]:
CORPUS_REVISION = "780f5516c4ae070761632d98ac3368f3ded09d35"
CORPUS_URL = f"https://codeload.github.com/roneysco/Fake.br-Corpus/zip/{CORPUS_REVISION}"
projectFolder = Path.cwd()
dataFolder = projectFolder / "data"
dataFolder.mkdir(exist_ok=True)
archivePath = dataFolder / f"Fake.br-Corpus-{CORPUS_REVISION}.zip"

if not archivePath.exists():
    temporaryPath = archivePath.with_suffix(".download")
    with urlopen(CORPUS_URL, timeout=120) as response, temporaryPath.open("wb") as target:
        while chunk := response.read(1024 * 1024):
            target.write(chunk)
    with ZipFile(temporaryPath) as archive:
        assert archive.testzip() is None, "ZIP corrompido; refaça o download."
    temporaryPath.replace(archivePath)

print("Corpus revision:", CORPUS_REVISION)
print("Archive SHA256:", hashlib.sha256(archivePath.read_bytes()).hexdigest())

In [ ]:
metadataColumns = [
    "autor", "link", "categoria", "data_publicacao",
    "num_tokens", "num_palavras", "num_types", "num_links", "num_maiusculas",
    "num_verbos", "num_verbos_subj_imp", "num_substantivos", "num_adjetivos",
    "num_adverbios", "num_verbos_modais", "num_pron_1_2_sing", "num_pron_1_plural",
    "num_pronomes", "pausalidade", "num_caracteres", "tam_medio_sentenca",
    "tam_medio_palavra", "pct_erros_ortograficos", "emotividade", "diversidade",
]

In [ ]:
def loadNewsTexts(archive, folder, label):
    records = []
    for name in sorted(archive.namelist()):
        if f"/full_texts/{folder}/" not in name or not name.endswith(".txt"):
            continue
        articleId = Path(name).stem + ("t" if label == 0 else "")
        records.append({
            "id": articleId, "text": archive.read(name).decode("utf-8"),
            "label": label, "sourceClass": folder,
        })
    assert records, f"Nenhum texto encontrado em {folder}"
    return pd.DataFrame(records)


def loadNewsMetadata(archive, folder, label):
    records = []
    for name in sorted(archive.namelist()):
        if f"/full_texts/{folder}-meta-information/" not in name or not name.endswith("-meta.txt"):
            continue
        values = [line.strip() for line in archive.read(name).decode("utf-8").splitlines()]
        assert len(values) == len(metadataColumns), f"Esquema inesperado em {name}: {len(values)} linhas"
        articleId = Path(name).name.removesuffix("-meta.txt") + ("t" if label == 0 else "")
        records.append({**dict(zip(metadataColumns, values)), "id": articleId, "metadataLabel": label})
    assert records, f"Nenhum metadado encontrado em {folder}"
    return pd.DataFrame(records)

In [ ]:
with ZipFile(archivePath) as archive:
    textsFrame = pd.concat([
        loadNewsTexts(archive, "fake", 1), loadNewsTexts(archive, "true", 0),
    ], ignore_index=True)
    metadataFrame = pd.concat([
        loadNewsMetadata(archive, "fake", 1), loadNewsMetadata(archive, "true", 0),
    ], ignore_index=True)

assert textsFrame["id"].is_unique and metadataFrame["id"].is_unique
assert set(textsFrame["id"]) == set(metadataFrame["id"]), "Texto/metadados sem correspondência"
newsFrame = textsFrame.merge(metadataFrame, on="id", validate="one_to_one", indicator=True)
assert newsFrame["_merge"].eq("both").all()
assert newsFrame["label"].eq(newsFrame["metadataLabel"]).all()
newsFrame = newsFrame.drop(columns=["_merge", "metadataLabel"])
assert newsFrame["text"].str.strip().ne("").all()
print(f"{len(newsFrame):,} notícias carregadas; textos e metadados correspondem 1:1.")

## 4. Contrato dos labels

**0 = True; 1 = Fake.** O rótulo forma as partições e permite avaliação externa.
Não entra como feature. A origem nas pastas também é verificada.

In [ ]:
labelNames = {0: "True", 1: "Fake"}
assert labelNames == {0: "True", 1: "Fake"}
assert set(newsFrame["label"].unique()) == {0, 1}
assert newsFrame.loc[newsFrame["label"].eq(0), "sourceClass"].eq("true").all()
assert newsFrame.loc[newsFrame["label"].eq(1), "sourceClass"].eq("fake").all()
display(newsFrame.groupby(["label", "sourceClass"]).size().rename("newsCount").to_frame())

## 5. Preparação dos metadados

Preservamos todos os campos brutos em `rawNewsFrame` e todas as contagens numéricas
em `newsFrame`. Valores não numéricos viram NaN com contagem explícita; nenhum
ausente é preenchido antes do treino. `tem_autor` segue a lógica do original:
ausência para string vazia, `None`, `none` ou `NULL`. Não é uma medida de credibilidade.

In [ ]:
rawNewsFrame = newsFrame.copy(deep=True)
numericMetadataColumns = metadataColumns[4:]
convertedMetadata = newsFrame[numericMetadataColumns].apply(pd.to_numeric, errors="coerce")
conversionMissing = convertedMetadata.isna().sum().rename("missingAfterNumericConversion")
display(conversionMissing.to_frame())
newsFrame[numericMetadataColumns] = convertedMetadata
newsFrame["tem_autor"] = (~newsFrame["autor"].fillna("").astype(str).str.strip().isin(
    ["", "None", "none", "NULL"]
)).astype(int)

## 6. Extração no texto efetivamente utilizado

Normalização Unicode NFKC, remoção do BOM inicial e primeiros CHARACTER_LIMIT
caracteres, incluindo espaços/pontuação. Textos menores permanecem menores, sem
preenchimento. O corte pode dividir palavras/frases. Palavras são sequências de
letras com hífen/apóstrofo interno; tokens também incluem números e pontuação.
Tipos são palavras distintas ignorando caixa. TTR = tipos/tokens; diversidade =
tipos/palavras. Maiúsculas conta palavras totalmente maiúsculas com mais de uma
letra. Links são URLs http(s) presentes no corpo. Estas definições explícitas
não pretendem reproduzir o extrator desconhecido dos metadados históricos.

As demais contagens linguísticas são marcadas ausentes na cópia de features,
não imputadas nem selecionadas pelo modelo; originais ficam em rawNewsFrame e
newsFrame. Autor permanece como metadado válido da notícia inteira.

In [ ]:
def calculateRatio(numerator, denominator):
    numerator = pd.to_numeric(numerator, errors="coerce").astype(float)
    denominator = pd.to_numeric(denominator, errors="coerce").astype(float)
    safeDenominator = denominator.where(denominator.gt(0) & np.isfinite(denominator))
    return numerator.div(safeDenominator).replace([np.inf, -np.inf], np.nan)

In [ ]:
import re
import unicodedata

CHARACTER_LIMIT = 300
numericMetadataColumns = metadataColumns[4:]
anomalyColumns = ["tem_autor", "typeTokenRatio", "linkDensity", "punctuationDensity", "uppercaseRatio", "diversidade"]
wordPattern = re.compile(r"[^\W\d_]+(?:['’\-][^\W\d_]+)*", re.UNICODE)
tokenPattern = re.compile(r"[^\W\d_]+(?:['’\-][^\W\d_]+)*|\d+(?:[.,]\d+)*|[^\w\s]", re.UNICODE)


def measureText(text, characterLimit):
    normalized = unicodedata.normalize("NFKC", text).lstrip("\ufeff")
    if characterLimit is not None:
        if not isinstance(characterLimit, int) or characterLimit < 1:
            raise ValueError("Limite deve ser inteiro positivo ou None.")
        normalized = normalized[:characterLimit]
    words = wordPattern.findall(normalized)
    return {"text": normalized, "num_palavras": len(words),
            "num_tokens": len(tokenPattern.findall(normalized)),
            "num_types": len({word.casefold() for word in words}),
            "num_links": len(re.findall(r"https?://\S+", normalized, flags=re.IGNORECASE)),
            "num_maiusculas": sum(word.isupper() and len(word) > 1 for word in words),
            "num_caracteres": len(normalized)}


def extractAnomalyFeatures(newsFrame, characterLimit=CHARACTER_LIMIT):
    featuresFrame = newsFrame.copy(deep=True)
    featuresFrame["num_palavras_original"] = newsFrame["num_palavras"]
    featuresFrame["text_original"] = newsFrame["text"]
    featuresFrame[numericMetadataColumns] = np.nan
    measurements = pd.DataFrame([measureText(text, characterLimit) for text in newsFrame["text"]], index=newsFrame.index)
    for column in measurements:
        featuresFrame[column] = measurements[column]
    featuresFrame["typeTokenRatio"] = calculateRatio(featuresFrame["num_types"], featuresFrame["num_tokens"])
    featuresFrame["diversidade"] = calculateRatio(featuresFrame["num_types"], featuresFrame["num_palavras"])
    featuresFrame["linkDensity"] = calculateRatio(featuresFrame["num_links"], featuresFrame["num_palavras"])
    featuresFrame["punctuationDensity"] = calculateRatio(featuresFrame["num_tokens"] - featuresFrame["num_palavras"], featuresFrame["num_tokens"])
    featuresFrame["uppercaseRatio"] = calculateRatio(featuresFrame["num_maiusculas"], featuresFrame["num_palavras"])
    return featuresFrame

In [ ]:
featuresFrame = extractAnomalyFeatures(newsFrame)
featuresFrame[anomalyColumns] = featuresFrame[anomalyColumns].replace([np.inf, -np.inf], np.nan)
display(featuresFrame[anomalyColumns].isna().sum().rename("missingCount").to_frame())
print("Limite de caracteres:", CHARACTER_LIMIT)
display(featuresFrame.groupby("label")[["num_palavras_original", "num_palavras", "num_caracteres"]].agg(["min", "median", "max"]))
print("Textos menores que o limite:", int(featuresFrame["num_caracteres"].lt(CHARACTER_LIMIT).sum()))

## 7. Sanity checks

As seis features utilizam o prefixo; a presença de autor vem dos metadados.
Contagens originais ficam preservadas para auditoria e não alimentam o modelo.

In [ ]:
assert "num_palavras" not in anomalyColumns
assert "label" not in anomalyColumns and "id" not in anomalyColumns
assert len(anomalyColumns) == len(set(anomalyColumns)) == 6
assert featuresFrame["id"].is_unique
assert featuresFrame["num_caracteres"].le(CHARACTER_LIMIT).all()
assert featuresFrame["num_palavras_original"].equals(newsFrame["num_palavras"])
assert featuresFrame["num_verbos"].isna().all()
assert not np.isinf(featuresFrame[anomalyColumns].to_numpy(dtype=float)).any()
assert measureText("casa " * 100, 300)["num_palavras"] == 60
assert measureText("casa " * 100, 300)["num_caracteres"] == 300
print("Contagens do prefixo verificadas; num_palavras não entra no treinamento.")

## 8. Análise estatística

Estatísticas descritivas por label, sem substituir vetores individuais por médias.
Esta inspeção do corpus completo atende ao protocolo exploratório: não usamos
seus resultados para selecionar features, ajustar hiperparâmetros ou threshold.
Qualquer escolha futura guiada por estes resultados exige nova avaliação independente.
Correlações de Pearson com comprimento são mostradas no total e por classe para
evitar confundir efeitos de classe e de tamanho. NaN em correlação pode indicar
feature constante. Mantemos as seis features recalculadas, inclusive possíveis redundâncias.

In [ ]:
statisticsRecords = []
for label, group in featuresFrame.groupby("label", sort=True):
    for feature in anomalyColumns:
        values = group[feature]
        q1, q3 = values.quantile([0.25, 0.75])
        statisticsRecords.append({
            "label": label, "class": labelNames[label], "feature": feature,
            "count": values.count(), "mean": values.mean(), "median": values.median(),
            "std": values.std(), "min": values.min(), "Q1": q1, "Q3": q3,
            "IQR": q3 - q1, "max": values.max(), "missingCount": values.isna().sum(),
        })
featureStatistics = pd.DataFrame(statisticsRecords).set_index(["label", "class", "feature"])
display(featureStatistics)

lengthColumns = ["num_palavras", "num_tokens"]
lengthCorrelations = pd.concat({
    name: group[anomalyColumns + lengthColumns].corr().loc[anomalyColumns, lengthColumns]
    for name, group in [
        ("All", featuresFrame),
        ("True (0)", featuresFrame.loc[featuresFrame["label"].eq(0)]),
        ("Fake (1)", featuresFrame.loc[featuresFrame["label"].eq(1)]),
    ]
}, names=["group", "feature"])
display(lengthCorrelations)
featureCorrelations = featuresFrame[anomalyColumns].corr()
display(featureCorrelations.round(3))
display(lengthCorrelations.xs("typeTokenRatio", level="feature"))
print("Pearson TTR vs diversidade:", featureCorrelations.loc["typeTokenRatio", "diversidade"])

## 9. Train / Validation / Test

True: 60% treino, 20% validação, 20% teste. Fake: 50% validação, 50% teste.
Todas as divisões usam `random_state=42`. Verificamos IDs exclusivos e cobertura
integral do corpus. **Nenhuma notícia Fake participa de fit ou calibração.**

Limitação do protocolo solicitado: a divisão é por notícia, não por assunto,
fonte ou data. O corpus possui pares True/Fake com o mesmo número-base; IDs
`123t` e `123` são notícias distintas, mas podem tratar do mesmo assunto em
partições diferentes. IDs exclusivos não demonstram independência temática.

In [ ]:
normalFrame = featuresFrame.loc[featuresFrame["label"].eq(0)].copy()
fakeFrame = featuresFrame.loc[featuresFrame["label"].eq(1)].copy()
normalTrainFrame, normalHoldoutFrame = train_test_split(
    normalFrame, train_size=0.60, random_state=RANDOM_STATE,
)
normalValidationFrame, normalTestFrame = train_test_split(
    normalHoldoutFrame, test_size=0.50, random_state=RANDOM_STATE,
)
fakeValidationFrame, fakeTestFrame = train_test_split(
    fakeFrame, test_size=0.50, random_state=RANDOM_STATE,
)
partitions = {
    "normalTrain": normalTrainFrame, "normalValidation": normalValidationFrame,
    "normalTest": normalTestFrame, "fakeValidation": fakeValidationFrame,
    "fakeTest": fakeTestFrame,
}
for name, frame in partitions.items():
    assert not frame.empty and frame["id"].is_unique
    assert frame["label"].eq(0 if name.startswith("normal") else 1).all()
for (leftName, left), (rightName, right) in combinations(partitions.items(), 2):
    assert set(left["id"]).isdisjoint(right["id"]), f"IDs compartilhados: {leftName}/{rightName}"
assert set().union(*(set(frame["id"]) for frame in partitions.values())) == set(featuresFrame["id"])
assert sum(len(frame) for frame in partitions.values()) == len(featuresFrame)
assert normalTrainFrame["label"].eq(0).all()
display(pd.DataFrame([
    {"partition": name, "count": len(frame), "label": int(frame["label"].iloc[0])}
    for name, frame in partitions.items()
]).set_index("partition"))

## 10. Isolation Forest

Pipeline fixo: imputação pela mediana + 300 árvores, sem StandardScaler.
O único `.fit()` recebe exclusivamente `normalTrain`. Uma guarda rejeita Fake,
infinitos e features inteiramente ausentes no treino (evita remoção silenciosa
de colunas pelo imputer). Nem médias do corpus nem dados de validação/teste
participam do ajuste.

In [ ]:
def fitNormalOnly(pipeline, trainingFrame):
    if trainingFrame.empty or not trainingFrame["label"].eq(0).all():
        raise ValueError("Treino permitido apenas com notícias True (label == 0).")
    trainingFeatures = trainingFrame[anomalyColumns]
    if np.isinf(trainingFeatures.to_numpy(dtype=float)).any():
        raise ValueError("Features de treino contêm infinito.")
    emptyColumns = trainingFeatures.columns[trainingFeatures.isna().all()].tolist()
    if emptyColumns:
        raise ValueError(f"Features inteiramente ausentes em normalTrain: {emptyColumns}")
    return pipeline.fit(trainingFeatures)

In [ ]:
anomalyPipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("detector", IsolationForest(
        n_estimators=300, contamination="auto", random_state=42, n_jobs=-1,
    )),
])
fitNormalOnly(anomalyPipeline, normalTrainFrame)
assert list(anomalyPipeline.feature_names_in_) == anomalyColumns
assert anomalyPipeline["detector"].n_features_in_ == len(anomalyColumns)
np.testing.assert_allclose(
    anomalyPipeline["imputer"].statistics_, normalTrainFrame[anomalyColumns].median().to_numpy(),
)
print(f"Pipeline ajustado em {len(normalTrainFrame)} notícias True e 0 Fake.")
display(pd.Series(anomalyPipeline["imputer"].statistics_, index=anomalyColumns, name="normalTrainMedian").to_frame())

## 11. Anomaly scores

Invertemos `decision_function`: score menor = mais normal; maior = mais anômalo.
Scores podem ser negativos. Não são probabilidades e não são convertidos em
percentuais de falsidade. Fake validation é reservada para inspeção visual.

In [ ]:
def calculateAnomalyScore(pipeline, featuresFrame):
    return -pipeline.decision_function(featuresFrame)

In [ ]:
normalValidationScores = calculateAnomalyScore(anomalyPipeline, normalValidationFrame[anomalyColumns])
fakeValidationScores = calculateAnomalyScore(anomalyPipeline, fakeValidationFrame[anomalyColumns])
testFrame = pd.concat([normalTestFrame, fakeTestFrame], ignore_index=True)
testResultsFrame = testFrame[["id", "label"] + anomalyColumns].copy()
testResultsFrame["anomalyScore"] = calculateAnomalyScore(anomalyPipeline, testFrame[anomalyColumns])
assert np.isfinite(normalValidationScores).all()
assert np.isfinite(fakeValidationScores).all()
assert np.isfinite(testResultsFrame["anomalyScore"]).all()

## 12. Threshold

Regra fixa: **percentil 95 dos scores de normalValidation**. Aproximadamente 5%
das notícias True de validação ultrapassam esse valor; empates podem aumentar
a proporção porque usamos `>=`. A taxa no teste pode ser diferente. Nenhuma
Fake ou notícia de teste define o threshold; `contamination="auto"` não escolhe
este corte empírico. Não usamos `predict()` do detector.

In [ ]:
threshold = np.quantile(normalValidationScores, 0.95)
testResultsFrame["isAnomaly"] = testResultsFrame["anomalyScore"] >= threshold
normalValidationAnomalyRate = np.mean(normalValidationScores >= threshold)
print(f"Threshold (percentil 95 de normalValidation): {threshold:.6f}")
print(f"True de validação marcadas como anômalas: {normalValidationAnomalyRate:.2%}")
assert np.isfinite(threshold)
assert "isFake" not in testResultsFrame.columns
display(testResultsFrame[["id", "label", "anomalyScore", "isAnomaly"]].head())

## 13. Avaliação

Avaliação final em **normalTest + fakeTest**. Classe positiva = **Fake (1)**
somente para medir discriminação. Precision/Recall/F1 e matriz comparam o
rótulo externo com o sinal `isAnomaly`, sem transformá-lo em classificação factual.
ROC-AUC e Average Precision usam scores contínuos. Reportamos **Average Precision
(AP)** como resumo da curva PR, não a integral trapezoidal PR-AUC.
A prevalência de Fake no teste é o referencial de AP de uma ordenação aleatória;
ela não representa a prevalência real na web.

In [ ]:
scoreStatistics = testResultsFrame.groupby("label")["anomalyScore"].agg(["count", "mean", "median", "std", "min", "max"])
scoreStatistics.insert(0, "class", scoreStatistics.index.map(labelNames))
display(scoreStatistics)

testLabels = testResultsFrame["label"].to_numpy()
testScores = testResultsFrame["anomalyScore"].to_numpy()
testFlags = testResultsFrame["isAnomaly"].to_numpy(dtype=int)
confusionMatrix = confusion_matrix(testLabels, testFlags, labels=[0, 1])
trueNegative, falsePositive, falseNegative, truePositive = confusionMatrix.ravel()
falsePositiveRate = falsePositive / (trueNegative + falsePositive)
fakeDetectionRate = truePositive / (falseNegative + truePositive)
fakePrevalence = np.mean(testLabels == 1)
metrics = {
    "ROC-AUC": roc_auc_score(testLabels, testScores),
    "Average Precision (AP)": average_precision_score(testLabels, testScores),
    "Precision": precision_score(testLabels, testFlags, zero_division=0),
    "Recall": recall_score(testLabels, testFlags, zero_division=0),
    "F1": f1_score(testLabels, testFlags, zero_division=0),
    "True false positive rate": falsePositiveRate,
    "Fake detection rate": fakeDetectionRate,
    "Fake prevalence (AP baseline)": fakePrevalence,
}
display(pd.Series(metrics, name="value").to_frame())
print(f"Notícias Fake detectadas como anômalas: {fakeDetectionRate:.2%}")
print(f"Notícias True marcadas como anômalas (FPR): {falsePositiveRate:.2%}")
display(pd.DataFrame(
    confusionMatrix, index=["Actual True (0)", "Actual Fake (1)"],
    columns=["Within threshold", "Anomaly"],
))
assert confusionMatrix.sum() == len(testResultsFrame)
assert set(testResultsFrame["id"]) == set(normalTestFrame["id"]) | set(fakeTestFrame["id"])
assert np.isclose(fakeDetectionRate, metrics["Recall"])

## 14. Visualizações

True (0) em azul e Fake (1) em laranja. Histogramas de densidade usam intervalos
comuns porque o teste tem tamanhos de classe diferentes. O primeiro painel
mostra validação apenas para inspeção; todos os demais mostram o teste.
O threshold permanece o mesmo, escolhido exclusivamente em normalValidation.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10), constrained_layout=True)
trueScores = testResultsFrame.loc[testResultsFrame["label"].eq(0), "anomalyScore"]
fakeScores = testResultsFrame.loc[testResultsFrame["label"].eq(1), "anomalyScore"]

for axis, groups, title in [
    (axes[0, 0], [normalValidationScores, fakeValidationScores], "Validação: inspeção dos scores"),
    (axes[0, 1], [trueScores, fakeScores], "Teste: distribuição dos scores"),
]:
    bins = np.histogram_bin_edges(np.concatenate(groups), bins=40)
    for values, name, color in zip(groups, ["True (0)", "Fake (1)"], ["tab:blue", "tab:orange"]):
        axis.hist(values, bins=bins, density=True, alpha=0.5, label=name, color=color)
    axis.axvline(threshold, color="black", linestyle="--", label=f"Threshold = {threshold:.3f}")
    axis.set(title=title, xlabel="anomalyScore (maior = mais anômalo)", ylabel="Densidade")
    axis.legend(fontsize=8)

boxes = axes[0, 2].boxplot([trueScores, fakeScores], patch_artist=True)
for box, color in zip(boxes["boxes"], ["tab:blue", "tab:orange"]):
    box.set_facecolor(color)
    box.set_alpha(0.5)
axes[0, 2].set_xticks([1, 2], ["True (0)", "Fake (1)"])
axes[0, 2].axhline(threshold, color="black", linestyle="--", label="Threshold")
axes[0, 2].set(title="Teste: boxplot por label", ylabel="anomalyScore")
axes[0, 2].legend(fontsize=8)

rocFpr, rocTpr, _ = roc_curve(testLabels, testScores, pos_label=1)
axes[1, 0].plot(rocFpr, rocTpr, label=f"ROC-AUC = {metrics['ROC-AUC']:.3f}")
axes[1, 0].plot([0, 1], [0, 1], "k--", label="Referência aleatória")
axes[1, 0].scatter([falsePositiveRate], [fakeDetectionRate], color="black", label="Threshold fixo")
axes[1, 0].set(title="Teste: ROC (positivo = Fake)", xlabel="False positive rate (True)", ylabel="True positive rate (Fake)")
axes[1, 0].legend(fontsize=8)

prPrecision, prRecall, _ = precision_recall_curve(testLabels, testScores, pos_label=1)
axes[1, 1].plot(prRecall, prPrecision, label=f"AP = {metrics['Average Precision (AP)']:.3f}")
axes[1, 1].axhline(fakePrevalence, color="black", linestyle="--", label=f"Prevalência Fake = {fakePrevalence:.3f}")
axes[1, 1].scatter([metrics["Recall"]], [metrics["Precision"]], color="black", label="Threshold fixo")
axes[1, 1].set(title="Teste: Precision-Recall (positivo = Fake)", xlabel="Recall", ylabel="Precision", ylim=(0, 1.05))
axes[1, 1].legend(fontsize=8)

ConfusionMatrixDisplay(confusionMatrix, display_labels=["True (0)", "Fake (1)"]).plot(
    ax=axes[1, 2], colorbar=False, cmap="Blues", values_format="d",
)
axes[1, 2].set_xticks([0, 1], ["Dentro do padrão", "Anomalia"])
axes[1, 2].set(title="Teste: matriz após threshold", xlabel="Sinal do detector", ylabel="Rótulo real")
axes[1, 2].grid(False)
plt.show()

## 15. Casos extremos

Exemplos exclusivamente do teste: cinco True com maior score, cinco Fake com
maior score e cinco Fake com menor score. Todas as seis features são apresentadas;
não são atribuições causais ou importâncias do Isolation Forest. NaN exibido
na tabela corresponde à feature antes da imputação, preservada para auditoria.

In [ ]:
exampleColumns = ["id", "label", "anomalyScore", "isAnomaly"] + anomalyColumns
extremeCases = {
    "5 True com maior anomalyScore": testResultsFrame.loc[testResultsFrame["label"].eq(0)].nlargest(5, "anomalyScore"),
    "5 Fake com maior anomalyScore": testResultsFrame.loc[testResultsFrame["label"].eq(1)].nlargest(5, "anomalyScore"),
    "5 Fake com menor anomalyScore": testResultsFrame.loc[testResultsFrame["label"].eq(1)].nsmallest(5, "anomalyScore"),
}
for title, examples in extremeCases.items():
    display(Markdown(f"**{title}**"))
    display(examples[exampleColumns].reset_index(drop=True))

## 16. Conclusão

Resumo calculado a partir desta execução, sem ajuste posterior de modelo ou
threshold. AUC resume ordenação; Recall/FPR resumem o corte escolhido. Nem uma
AUC acima de 0,5 demonstra significância estatística, nem um score alto prova
falsidade. Não estimamos intervalos de confiança nesta V1.

In [ ]:
display(Markdown(f"""
Foram treinadas **{len(normalTrainFrame)} notícias True e nenhuma Fake**,
usando até **{CHARACTER_LIMIT} caracteres** e seis features recalculadas.
No teste: **{truePositive}/{len(fakeTestFrame)} Fake detectadas** e
**{falsePositive}/{len(normalTestFrame)} True sinalizadas**.
ROC-AUC: **{metrics['ROC-AUC']:.4f}**; recall **{fakeDetectionRate:.2%}**;
FPR **{falsePositiveRate:.2%}**. Corte: **{threshold:.6f}**, calibrado apenas em True.

num_palavras permanece na extração, mas está fora da matriz de treino.
Correlações por classe estão na seção 8. Caracteres fixos não fixam palavras;
o corte também remove conteúdo e pode cortar frases. Não ajustamos o corte pelo teste.
O controle da seção 17 usa o mesmo extrator sobre o texto completo.
**Anomalia não comprova falsidade.**
"""))

## 17. Controle de texto completo com o mesmo extrator

Recalculamos as mesmas seis features no texto completo e reutilizamos exatamente
os IDs das partições acima. Cada modelo tem seu corte calibrado na própria
validação True. Esta comparação mantém constantes extrator, features, IDs e
hiperparâmetros. O truncamento muda o conteúdo disponível, portanto não demonstra
isoladamente um efeito causal da quantidade de palavras.

In [ ]:
from sklearn.base import clone

fullFeaturesFrame = extractAnomalyFeatures(newsFrame, characterLimit=None).set_index("id", drop=False)
fullTrainFrame = fullFeaturesFrame.loc[normalTrainFrame["id"]]
fullValidationFrame = fullFeaturesFrame.loc[normalValidationFrame["id"]]
fullTestFrame = fullFeaturesFrame.loc[testFrame["id"]]
assert fullTrainFrame["id"].tolist() == normalTrainFrame["id"].tolist()
assert fullTestFrame["label"].tolist() == testLabels.tolist()
fullPipeline = clone(anomalyPipeline)
fitNormalOnly(fullPipeline, fullTrainFrame)
fullValidationScores = calculateAnomalyScore(fullPipeline, fullValidationFrame[anomalyColumns])
fullThreshold = np.quantile(fullValidationScores, 0.95)
fullTestScores = calculateAnomalyScore(fullPipeline, fullTestFrame[anomalyColumns])
fullFlags = fullTestScores >= fullThreshold
lengthComparison = pd.DataFrame([
    {"modelo": "Texto completo: mesmo extrator, seis features", "ROC-AUC": roc_auc_score(testLabels, fullTestScores),
     "Fake detectadas": int(np.sum(fullFlags & (testLabels == 1))), "True sinalizadas": int(np.sum(fullFlags & (testLabels == 0))),
     "corte": float(fullThreshold)},
    {"modelo": f"Primeiros {CHARACTER_LIMIT} caracteres: seis features", "ROC-AUC": metrics["ROC-AUC"],
     "Fake detectadas": int(truePositive), "True sinalizadas": int(falsePositive), "corte": float(threshold)},
])
display(lengthComparison)
display(pd.concat({
    "Completo - treino True": fullTrainFrame[anomalyColumns + ["num_palavras"]].corr()["num_palavras"].drop("num_palavras"),
    "Truncado - treino True": normalTrainFrame[anomalyColumns + ["num_palavras"]].corr()["num_palavras"].drop("num_palavras"),
}, axis=1))
assert "num_palavras" not in fullPipeline.feature_names_in_
assert "num_palavras" not in anomalyPipeline.feature_names_in_
print("Comparação concluída, sem seleção de modelo ou ajuste de corte pelo teste.")

## 18. Todas as janelas: cobertura, treino e agregação

Os IDs de treino/validação/teste já estão definidos nos controles acima.
Criamos janelas somente dentro de cada notícia; nunca separamos janelas da
mesma notícia entre treino e teste. O texto é normalizado em NFKC; espaços,
pontuação e fragmento final são mantidos. Cortes podem dividir palavras/frases.

In [ ]:
def splitWindows(text, windowSize=300):
    if not isinstance(windowSize, int) or isinstance(windowSize, bool) or windowSize < 1:
        raise ValueError("windowSize deve ser um inteiro positivo.")
    normalized = unicodedata.normalize("NFKC", text).lstrip("\ufeff")
    if not normalized:
        raise ValueError("Texto vazio não permite criar janelas.")
    return [normalized[start:start + windowSize] for start in range(0, len(normalized), windowSize)]

In [ ]:
def aggregateWindowScores(scoredWindows):
    values = scoredWindows[["windowCharacters", "windowScore"]].to_numpy(dtype=float)
    if scoredWindows.empty or not np.isfinite(values).all() or (values[:, 0] <= 0).any():
        raise ValueError("Scores finitos e pesos positivos são obrigatórios.")
    weighted = scoredWindows.assign(weightedScore=scoredWindows["windowScore"] * scoredWindows["windowCharacters"])
    result = weighted.groupby("id", sort=False).agg(
        totalWeightedScore=("weightedScore", "sum"), totalCharacters=("windowCharacters", "sum"),
        windowCount=("windowScore", "size"), maximumWindowScore=("windowScore", "max"),
    ).reset_index()
    result["anomalyScore"] = result["totalWeightedScore"] / result["totalCharacters"]
    return result.drop(columns="totalWeightedScore")

In [ ]:
def selectTrainingWindows(trainingWindows, seed=42):
    if trainingWindows.empty or not trainingWindows["label"].eq(0).all():
        raise ValueError("Seleção de treino exige exclusivamente notícias True.")
    generator = np.random.default_rng(seed)
    selected = []
    for articleId, group in trainingWindows.groupby("id", sort=False):
        weights = group["windowCharacters"].to_numpy(dtype=float)
        if not np.isfinite(weights).all() or (weights <= 0).any():
            raise ValueError("Comprimentos das janelas devem ser positivos e finitos.")
        position = generator.choice(len(group), p=weights / weights.sum())
        selected.append(group.iloc[[position]])
    return pd.concat(selected, ignore_index=True)

In [ ]:
def makeWindowFrame(articles, windowSize=300):
    if articles.empty or not articles["id"].is_unique:
        raise ValueError("São necessárias notícias com IDs únicos.")
    records = []
    for row in articles[["id", "label", "tem_autor", "text"]].itertuples(index=False):
        for windowIndex, chunk in enumerate(splitWindows(row.text, windowSize)):
            records.append({"id": row.id, "label": row.label, "tem_autor": row.tem_autor,
                            "text": chunk, "num_palavras": np.nan,
                            "windowIndex": windowIndex, "windowCharacters": len(chunk)})
    return extractAnomalyFeatures(pd.DataFrame(records), characterLimit=None)


def scoreWindowArticles(pipeline, windows, articles):
    if not articles["id"].is_unique or set(windows["id"]) != set(articles["id"]):
        raise ValueError("Janelas e notícias precisam cobrir os mesmos IDs únicos.")
    scored = windows[["id", "windowIndex", "windowCharacters", "num_palavras", "text"]].copy()
    scored["windowScore"] = calculateAnomalyScore(pipeline, windows[anomalyColumns])
    combined = aggregateWindowScores(scored)
    results = articles[["id", "label"]].merge(combined, on="id", validate="one_to_one", sort=False)
    return results, scored

In [ ]:
from sklearn.base import clone

WINDOW_SIZE = 300
rawById = newsFrame.set_index("id", drop=False)
windowPartitions = {}
for name, frame in partitions.items():
    windowPartitions[name] = makeWindowFrame(rawById.loc[frame["id"]], WINDOW_SIZE)
    assert set(windowPartitions[name]["id"]) == set(frame["id"])
for (leftName, left), (rightName, right) in combinations(windowPartitions.items(), 2):
    assert set(left["id"]).isdisjoint(right["id"])

selectedTrainingWindows = selectTrainingWindows(windowPartitions["normalTrain"], seed=RANDOM_STATE)
assert len(selectedTrainingWindows) == len(normalTrainFrame) == 2160
assert selectedTrainingWindows["id"].is_unique
assert set(selectedTrainingWindows["id"]) == set(normalTrainFrame["id"])
windowPipeline = clone(anomalyPipeline)
fitNormalOnly(windowPipeline, selectedTrainingWindows)
assert "num_palavras" not in windowPipeline.feature_names_in_

windowValidationResults, windowValidationDetails = scoreWindowArticles(
    windowPipeline, windowPartitions["normalValidation"], normalValidationFrame,
)
windowThreshold = float(np.quantile(windowValidationResults["anomalyScore"], 0.95))
windowTestFrame = pd.concat([windowPartitions["normalTest"], windowPartitions["fakeTest"]], ignore_index=True)
windowTestResults, windowTestDetails = scoreWindowArticles(windowPipeline, windowTestFrame, testFrame)
windowTestResults["isAnomaly"] = windowTestResults["anomalyScore"] >= windowThreshold
assert windowTestResults["id"].tolist() == testFrame["id"].tolist()
assert windowTestResults["label"].tolist() == testLabels.tolist()
assert len(windowTestResults) == 2520

windowCounts = pd.DataFrame([
    {"partição": name, "notícias": frame["id"].nunique(), "trechos disponíveis": len(frame),
     "trechos usados em fit": len(selectedTrainingWindows) if name == "normalTrain" else 0}
    for name, frame in windowPartitions.items()
])
display(windowCounts)
print("Corte sobre notícias True de validação:", windowThreshold)
print("FPR de validação:", float((windowValidationResults["anomalyScore"] >= windowThreshold).mean()))

## 19. Comparação no mesmo teste por notícia

Cada linha abaixo foi reexecutada neste notebook com as mesmas seis features,
os mesmos IDs, seed e hiperparâmetros. O modelo por janelas recebe um trecho
sorteado por notícia no fit, enquanto o controle truncado usa sempre a abertura.
Os cortes são calibrados separadamente em scores de notícias True de validação.

num_palavras não entra diretamente em nenhum modelo. Autor permanece em todos.
ROC-AUC mede ordenação; acurácia e recall dependem do corte e da prevalência.

In [ ]:
def summarizeNewsScores(name, labels, scores, cutoff):
    labels, scores = np.asarray(labels), np.asarray(scores)
    flags = scores >= cutoff
    tn, fp, fn, tp = confusion_matrix(labels, flags, labels=[0, 1]).ravel()
    return {"modelo": name, "Fake detectadas": int(tp), "Fake perdidas": int(fn),
            "True sinalizadas": int(fp), "True sem alerta": int(tn),
            "acurácia": float((tp + tn) / len(labels)), "recall": float(tp / (tp + fn)),
            "FPR": float(fp / (fp + tn)), "precision": float(tp / (tp + fp)) if tp + fp else 0.0,
            "F1": float(2 * tp / (2 * tp + fp + fn)), "ROC-AUC": float(roc_auc_score(labels, scores)),
            "AP": float(average_precision_score(labels, scores)), "corte": float(cutoff)}

windowComparison = pd.DataFrame([
    summarizeNewsScores("Texto completo - seis features", testLabels, fullTestScores, fullThreshold),
    summarizeNewsScores("Primeiros 300 caracteres - seis features", testLabels, testScores, threshold),
    summarizeNewsScores("Todos os trechos - média ponderada", testLabels, windowTestResults["anomalyScore"], windowThreshold),
])
display(windowComparison)

windowConfusion = confusion_matrix(testLabels, windowTestResults["isAnomaly"], labels=[0, 1])
windowFig, windowAxes = plt.subplots(1, 3, figsize=(17, 4), constrained_layout=True)
for label, group in windowTestResults.groupby("label"):
    windowAxes[0].hist(group["anomalyScore"], bins=40, density=True, alpha=0.5, label=labelNames[label])
windowAxes[0].axvline(windowThreshold, color="black", linestyle="--", label="Corte agregado")
windowAxes[0].set(title="Scores por notícia: todas as janelas", xlabel="Score agregado", ylabel="Densidade")
windowAxes[0].legend()
for name, scores in [("Completo", fullTestScores), ("Prefixo", testScores), ("Janelas", windowTestResults["anomalyScore"])]:
    fpr, tpr, _ = roc_curve(testLabels, scores)
    windowAxes[1].plot(fpr, tpr, label=f"{name}: {roc_auc_score(testLabels, scores):.4f}")
windowAxes[1].plot([0, 1], [0, 1], "k--")
windowAxes[1].set(title="ROC: mesma população de teste", xlabel="FPR", ylabel="Recall")
windowAxes[1].legend()
ConfusionMatrixDisplay(windowConfusion, display_labels=["True", "Fake"]).plot(ax=windowAxes[2], colorbar=False, values_format="d")
windowAxes[2].set_title("Janelas: decisão por notícia")
plt.show()

windowAudit = windowTestResults.merge(rawById[["id", "num_palavras"]].reset_index(drop=True), on="id", validate="one_to_one")
display(windowAudit.groupby("label")[["anomalyScore", "num_palavras", "windowCount"]].corr().loc[(slice(None), "anomalyScore"), :])
print("Correlação descritiva por classe, sem ajustar a agregação ou o corte com estes resultados.")

## 20. Resultado e limites

Todos os trechos contribuem para o score final, mas suas palavras são representadas
apenas por seis medidas; não há compreensão semântica nem verificação factual.
Scores máximos ficam disponíveis para auditoria e não alteram a decisão principal.
Uma média pode diluir um trecho isolado suspeito. Também permanecem efeitos de
fonte/autoria, tema, tamanho, posição e pares temáticos entre partições.
Não usamos os resultados para escolher outro agregador, semente ou cutoff.

In [ ]:
windowResult = windowComparison.iloc[-1]
display(Markdown(f"""
Foram treinadas **2160 notícias True e nenhuma Fake**, com um trecho sorteado por notícia.
Na validação e no teste, todas as janelas contribuem para a média ponderada por caracteres.
O teste possui **720 True e 1800 Fake**.

**Todos os trechos:** {int(windowResult['Fake detectadas'])}/1800 Fake detectadas;
{int(windowResult['True sinalizadas'])}/720 True sinalizadas por engano.
Acurácia **{windowResult['acurácia']:.2%}**; recall **{windowResult['recall']:.2%}**;
FPR **{windowResult['FPR']:.2%}**; ROC-AUC **{windowResult['ROC-AUC']:.4f}**.

Este resultado avalia combinação de medidas de escrita neste split do corpus.
**Anomalia não comprova falsidade.** A mudança inclui amostragem de treino e agregação,
não somente cobertura do texto; não demonstra superioridade estatística nem generalização.
"""))

## 21. Reprodução do baseline e diagnóstico dos falsos positivos

Esta seção usa somente os scores da reprodução windowed já executada. O corte é
calibrado nas 720 True de validação e a decisão continua `score >= corte`.
O diagnóstico lista todas as True do teste, deriva os falsos positivos pela
decisão, mostra a margem ao corte e as três janelas de maior score. A
contribuição é aritmética (`score_da_janela * caracteres / caracteres_totais`);
ela não é atribuição causal de uma feature.

In [ ]:
from urllib.parse import urlparse

singleValidation = windowValidationResults.set_index("id")["anomalyScore"].reindex(normalValidationFrame["id"]).to_numpy(dtype=float)
singleTest = windowTestResults.set_index("id")["anomalyScore"].reindex(testFrame["id"]).to_numpy(dtype=float)
singleCutoffs = {q: float(np.quantile(singleValidation, q)) for q in (0.95, 0.97)}

diagnosticWindows = windowTestDetails.merge(
    windowTestFrame[["id", "windowIndex", *anomalyColumns]],
    on=["id", "windowIndex"], how="left", validate="one_to_one",
)
diagnosticWindows["charStart"] = diagnosticWindows.groupby("id", sort=False)["windowCharacters"].cumsum() - diagnosticWindows["windowCharacters"]
diagnosticWindows["charEnd"] = diagnosticWindows["charStart"] + diagnosticWindows["windowCharacters"]
diagnosticArticleFrame = testFrame[["id", "label", "tem_autor", "categoria", "link", "num_palavras"]].copy()
diagnosticArticleFrame["articleScore"] = singleTest
diagnosticArticleFrame["cutoff"] = singleCutoffs[0.95]
diagnosticArticleFrame["margin"] = diagnosticArticleFrame["articleScore"] - diagnosticArticleFrame["cutoff"]
diagnosticArticleFrame["isAnomaly"] = diagnosticArticleFrame["articleScore"] >= diagnosticArticleFrame["cutoff"]
diagnosticArticleFrame["sourceDomain"] = diagnosticArticleFrame["link"].map(lambda value: urlparse(str(value)).hostname if pd.notna(value) and str(value).strip() else None)

diagnosticArticles = []
for row in diagnosticArticleFrame.loc[diagnosticArticleFrame["label"].eq(0)].itertuples(index=False):
    windows = diagnosticWindows.loc[diagnosticWindows["id"].eq(row.id)].sort_values(["windowScore", "windowIndex"], ascending=[False, True])
    totalCharacters = float(windows["windowCharacters"].sum())
    topWindows = []
    for window in windows.head(3).itertuples(index=False):
        topWindows.append({
            "windowIndex": int(window.windowIndex), "charStart": int(window.charStart), "charEnd": int(window.charEnd),
            "charCount": int(window.windowCharacters), "featureValues": {feature: (None if pd.isna(getattr(window, feature)) else float(getattr(window, feature))) for feature in anomalyColumns},
            "windowScore": float(window.windowScore),
            "weightedContribution": float(window.windowScore * window.windowCharacters / totalCharacters),
        })
    diagnosticArticles.append({
        "articleId": str(row.id), "label": int(row.label), "authorPresent": None if pd.isna(row.tem_autor) else bool(row.tem_autor),
        "category": None if pd.isna(row.categoria) else row.categoria, "sourceDomain": row.sourceDomain,
        "originalWordCount": None if pd.isna(row.num_palavras) else int(row.num_palavras), "windowCount": int(len(windows)),
        "articleScore": float(row.articleScore), "cutoff": float(row.cutoff), "margin": float(row.margin),
        "isAnomaly": bool(row.isAnomaly), "topWindows": topWindows,
    })

def diagnosticGroups(rows, dimension):
    frame = pd.DataFrame(rows)
    result = []
    for value, group in frame.groupby(dimension, dropna=False, sort=False):
        denominator = int(len(group))
        alerted = int(group["isAnomaly"].sum())
        result.append({"dimension": dimension, "group": None if pd.isna(value) else value, "missing": bool(pd.isna(value)),
                       "alerted": alerted, "notAlerted": denominator - alerted, "denominator": denominator,
                       "alertRate": float(alerted / denominator) if denominator else None})
    return result

trueDiagnosticRows = [row for row in diagnosticArticles if row["label"] == 0]
diagnostic = {
    "articles": diagnosticArticles,
    "falsePositives": [row for row in trueDiagnosticRows if row["isAnomaly"]],
    "groupComparisons": sum((diagnosticGroups(trueDiagnosticRows, dimension) for dimension in ("authorPresent", "category", "sourceDomain", "windowCount")), []),
    "distributions": {
        "margins": pd.Series([row["margin"] for row in trueDiagnosticRows]).describe(percentiles=[.25, .5, .75]).to_dict(),
        "windowScores": diagnosticWindows["windowScore"].describe(percentiles=[.25, .5, .75]).to_dict(),
    },
}
display(pd.DataFrame([{
    "True no teste": len(trueDiagnosticRows), "Falsos positivos": len(diagnostic["falsePositives"]),
    "corte q95": singleCutoffs[0.95], "margem mínima": min(row["margin"] for row in trueDiagnosticRows),
    "margem máxima": max(row["margin"] for row in trueDiagnosticRows),
}]))
display(pd.DataFrame([{
    "articleId": row["articleId"], "score": row["articleScore"], "margem": row["margin"],
    "janelas": row["windowCount"], "autor": row["authorPresent"], "categoria": row["category"],
} for row in diagnostic["falsePositives"]]).sort_values("margem", ascending=True))

## 22. Preparação dos scores para comparação

Os três pipelines abaixo são apenas a preparação fixa prevista para as
próximas etapas: uma janela True por artigo, sementes 42/43/44, estimador com
seed 42 e agregação ponderada por caracteres. Nenhum parâmetro é escolhido com
os rótulos do teste.

In [ ]:
ensembleValidationMembers = []
ensembleTestMembers = []
ensembleSelections = {}
for samplingSeed in (42, 43, 44):
    selected = selectTrainingWindows(windowPartitions["normalTrain"], seed=samplingSeed)
    ensembleSelections[samplingSeed] = selected[["id", "windowIndex", "windowCharacters", "label"]].copy()
    memberPipeline = clone(anomalyPipeline)
    fitNormalOnly(memberPipeline, selected)
    validationMember, _ = scoreWindowArticles(memberPipeline, windowPartitions["normalValidation"], normalValidationFrame)
    testMember, _ = scoreWindowArticles(memberPipeline, windowTestFrame, testFrame)
    ensembleValidationMembers.append(validationMember[["id", "anomalyScore"]].rename(columns={"anomalyScore": f"score_{samplingSeed}"}))
    ensembleTestMembers.append(testMember[["id", "anomalyScore"]].rename(columns={"anomalyScore": f"score_{samplingSeed}"}))

ensembleValidationFrame = ensembleValidationMembers[0]
for member in ensembleValidationMembers[1:]:
    ensembleValidationFrame = ensembleValidationFrame.merge(member, on="id", validate="one_to_one")
ensembleValidationFrame["ensembleScore"] = ensembleValidationFrame[["score_42", "score_43", "score_44"]].mean(axis=1)
ensembleTestFrame = ensembleTestMembers[0]
for member in ensembleTestMembers[1:]:
    ensembleTestFrame = ensembleTestFrame.merge(member, on="id", validate="one_to_one")
ensembleTestFrame["ensembleScore"] = ensembleTestFrame[["score_42", "score_43", "score_44"]].mean(axis=1)
ensembleValidation = ensembleValidationFrame.set_index("id")["ensembleScore"].reindex(normalValidationFrame["id"]).to_numpy(dtype=float)
ensembleTest = ensembleTestFrame.set_index("id")["ensembleScore"].reindex(testFrame["id"]).to_numpy(dtype=float)
ensembleCutoffs = {q: float(np.quantile(ensembleValidation, q)) for q in (0.95, 0.97)}
assert all(len(selection) == 2160 and selection["id"].is_unique and selection["label"].eq(0).all() for selection in ensembleSelections.values())
display(pd.DataFrame([{"seed": seed, "selectedTrueArticles": len(selection), "uniqueWindows": selection[["id", "windowIndex"]].drop_duplicates().shape[0]} for seed, selection in ensembleSelections.items()]))

## 23. Quatro cortes fixos e comparação no mesmo teste

Os cortes q95/q97 são derivados somente das True de validação. A tabela salva
TP/FP/TN/FN, métricas dependentes da decisão, métricas de ordenação e o saldo
`FP evitados - FN adicionais` em relação a `single_q95`. Não há vencedor
automático: a execução é exploratória e o teste histórico já foi inspecionado.

In [ ]:
from sklearn.metrics import average_precision_score, roc_auc_score

caseScores = {"single": singleTest, "ensemble": ensembleTest}
caseCutoffs = {"single_q95": singleCutoffs[0.95], "ensemble_q95": ensembleCutoffs[0.95], "single_q97": singleCutoffs[0.97], "ensemble_q97": ensembleCutoffs[0.97]}
caseValidationFpr = {"single_q95": float(np.mean(singleValidation >= singleCutoffs[0.95])), "single_q97": float(np.mean(singleValidation >= singleCutoffs[0.97])),
                     "ensemble_q95": float(np.mean(ensembleValidation >= ensembleCutoffs[0.95])), "ensemble_q97": float(np.mean(ensembleValidation >= ensembleCutoffs[0.97]))}
caseSpecs = [("single_q95", "single", .95), ("ensemble_q95", "ensemble", .95), ("single_q97", "single", .97), ("ensemble_q97", "ensemble", .97)]

def _safeRatio(numerator, denominator):
    return None if denominator == 0 else float(numerator / denominator)

comparisonRows = []
caseDecisions = {}
for caseKey, modelKey, quantile in caseSpecs:
    scores = np.asarray(caseScores[modelKey], dtype=float)
    decisions = scores >= caseCutoffs[caseKey]
    caseDecisions[caseKey] = decisions
    tn = int(np.sum((testLabels == 0) & ~decisions)); fp = int(np.sum((testLabels == 0) & decisions))
    fn = int(np.sum((testLabels == 1) & ~decisions)); tp = int(np.sum((testLabels == 1) & decisions))
    row = {"modelKey": caseKey, "model": modelKey, "quantile": quantile, "cutoff": caseCutoffs[caseKey], "validationFPR": caseValidationFpr[caseKey],
           "TP": tp, "FP": fp, "TN": tn, "FN": fn, "total": len(testLabels), "accuracy": _safeRatio(tp + tn, len(testLabels)),
           "recall": _safeRatio(tp, tp + fn), "fpr": _safeRatio(fp, fp + tn), "precision": _safeRatio(tp, tp + fp), "f1": _safeRatio(2 * tp, 2 * tp + fp + fn),
           "rocAuc": float(roc_auc_score(testLabels, scores)), "averagePrecision": float(average_precision_score(testLabels, scores)), "decisionCount": int(decisions.sum()),
           "fakeDetected": tp, "fakeTotal": int(np.sum(testLabels == 1)), "fakeMissed": fn,
           "trueSignaled": fp, "trueTotal": int(np.sum(testLabels == 0)), "trueWithoutAlert": tn,
           "correct": tp + tn}
    comparisonRows.append(row)
baselineRow = comparisonRows[0]
for row in comparisonRows:
    fpAvoided = baselineRow["FP"] - row["FP"]; fnAdditional = row["FN"] - baselineRow["FN"]
    row["deltaVsSingleQ95"] = {"TP": row["TP"] - baselineRow["TP"], "FP": row["FP"] - baselineRow["FP"], "TN": row["TN"] - baselineRow["TN"], "FN": row["FN"] - baselineRow["FN"],
                                "fpAvoided": fpAvoided, "fnAdditional": fnAdditional, "netFpAvoidedMinusFnAdditional": fpAvoided - fnAdditional}

comparisonTransitions = []
for index, articleId in enumerate(testFrame["id"].astype(str)):
    baselineDecision = bool(caseDecisions["single_q95"][index])
    transitions = {}
    for caseKey, _, _ in caseSpecs:
        current = bool(caseDecisions[caseKey][index])
        transitions[caseKey] = "baseline" if caseKey == "single_q95" else ("mantido" if current == baselineDecision else ("adicionado" if current else "removido"))
    comparisonTransitions.append({"articleId": articleId, "label": int(testLabels[index]), "decisions": {case: bool(caseDecisions[case][index]) for case, _, _ in caseSpecs}, "transition": transitions})
evaluationReport = {"comparisons": comparisonRows, "transitions": comparisonTransitions,
                    "ordering": [{"model": model, "rocAuc": next(row["rocAuc"] for row in comparisonRows if row["model"] == model),
                                  "averagePrecision": next(row["averagePrecision"] for row in comparisonRows if row["model"] == model),
                                  "note": "ROC-AUC/AP medem ordenação; corte governa decisão."} for model in ("single", "ensemble")],
                    "status": "exploratory_test_inspected",
                    "warnings": ["Anomalia não comprova falsidade.", "Não há seleção automática de cenário.", "Colab não foi testado nesta execução local."]}
summaryRows = [{
    "cenário": row["modelKey"],
    "Fake detectadas corretamente": f"{row['fakeDetected']}/{row['fakeTotal']}",
    "Fake perdidas": row["fakeMissed"],
    "True sinalizadas": f"{row['trueSignaled']}/{row['trueTotal']}",
    "True sem alerta": row["trueWithoutAlert"],
    "Acertos": f"{row['correct']}/{row['total']}",
    "Acurácia": row["accuracy"],
} for row in comparisonRows]
display(Markdown("### Resultado por cenário\n\n" + "\n".join(
    f"- **{row['modelKey']}**: **{row['fakeDetected']}/{row['fakeTotal']} Fake detectadas corretamente**, "
    f"{row['fakeMissed']} Fake perdidas; **{row['trueSignaled']}/{row['trueTotal']} True sinalizadas**, "
    f"{row['trueWithoutAlert']} True sem alerta; **{row['correct']}/{row['total']} acertos** "
    f"(acurácia {row['accuracy']:.2%})."
    for row in comparisonRows
)))
display(pd.DataFrame(summaryRows))
display(pd.DataFrame(comparisonRows)[["modelKey", "cutoff", "TP", "FP", "TN", "FN", "accuracy", "recall", "fpr", "rocAuc", "averagePrecision"]])

## 24. Limites e execução

O score é uma medida de anomalia de seis características; não é probabilidade
de falsidade e não substitui revisão factual. A diferença entre q95 e q97
expressa uma decisão mais conservadora, não uma mudança de ordenação. Este
notebook roda em kernel local/Colab, mas a validação de upload/Run All no
Colab permanece **não testada**. O executor isolado que salva uma cópia
executada e um novo `artifacts/windowed-improvements/<run-id>/` pertence à
TASK-007, ainda pendente.

## 25. Hybrid rank, agregadores registrados e evidências

Os helpers abaixo são embutidos no notebook no momento da construção; nenhum módulo local é importado durante o runtime. Seeds, pesos, proporção aparada e thresholds são congelados antes de abrir o teste.

In [ ]:
"""Pure helpers for the pre-registered hybrid rank experiment.

The module deliberately does not load the corpus or fit a model.  It operates
on already produced scores and windows so that the statistical layer cannot
silently change the data protocol or use evaluation labels for calibration.
"""

from __future__ import annotations

import math
from itertools import combinations
from typing import Any, Mapping, Sequence

import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, roc_auc_score


EXPECTED_MEMBER_SCORE_COLUMNS = ("score_42", "score_43", "score_44")
TRIM_PROPORTION = 0.10


def _vector(values: Any, name: str, *, allow_empty: bool = False) -> np.ndarray:
    array = np.asarray(values, dtype=float)
    if array.ndim != 1 or (not allow_empty and array.size == 0):
        raise ValueError(f"{name} deve ser um vetor não vazio.")
    if not np.isfinite(array).all():
        raise ValueError(f"{name} deve conter somente valores finitos.")
    return array


def ecdf_percentile(reference_scores: Sequence[float], values: Sequence[float]) -> np.ndarray:
    """Map values to an inclusive empirical CDF built only from reference scores.

    P(X <= value) is used for ties.  The function never inspects labels and
    therefore callers can pass only the True calibration scores as the
    reference distribution.
    """

    reference = np.sort(_vector(reference_scores, "reference_scores"))
    observations = _vector(values, "values")
    ranks = np.searchsorted(reference, observations, side="right")
    return ranks.astype(float) / float(reference.size)


def align_by_id(
    left: pd.DataFrame,
    right: pd.DataFrame,
    *,
    left_columns: Sequence[str] = (),
    right_columns: Sequence[str] = (),
) -> pd.DataFrame:
    """Align two score tables by unique article ID while preserving left order."""

    if not isinstance(left, pd.DataFrame) or not isinstance(right, pd.DataFrame):
        raise ValueError("left e right devem ser DataFrames.")

    def id_name(frame: pd.DataFrame) -> str:
        if "articleId" in frame.columns:
            return "articleId"
        if "id" in frame.columns:
            return "id"
        raise ValueError("Cada tabela requer articleId ou id.")

    left_id, right_id = id_name(left), id_name(right)
    left_ids = left[left_id].astype(str)
    right_ids = right[right_id].astype(str)
    if left[left_id].isna().any() or right[right_id].isna().any():
        raise ValueError("IDs não podem ser nulos.")
    if left_ids.duplicated().any() or right_ids.duplicated().any():
        raise ValueError("IDs devem ser únicos em cada tabela.")
    if set(left_ids) != set(right_ids):
        raise ValueError("As tabelas não cobrem o mesmo conjunto de IDs.")

    selected_left = list(left_columns)
    selected_right = list(right_columns)
    missing_left = set(selected_left) - set(left.columns)
    missing_right = set(selected_right) - set(right.columns)
    if missing_left or missing_right:
        raise ValueError(f"Colunas ausentes: left={sorted(missing_left)}, right={sorted(missing_right)}")
    overlap = set(selected_left) & set(selected_right)
    if overlap:
        raise ValueError(f"Colunas duplicadas entre as tabelas: {sorted(overlap)}")

    left_part = pd.DataFrame({"articleId": left_ids.tolist()})
    for column in selected_left:
        left_part[column] = left[column].to_numpy()
    right_part = pd.DataFrame({"articleId": right_ids.tolist()})
    for column in selected_right:
        right_part[column] = right[column].to_numpy()
    return left_part.merge(right_part, on="articleId", how="left", sort=False, validate="one_to_one")


def _weighted_inputs(scores: Sequence[float], char_counts: Sequence[float]) -> tuple[np.ndarray, np.ndarray]:
    values = _vector(scores, "scores")
    weights = _vector(char_counts, "char_counts")
    if values.size != weights.size:
        raise ValueError("scores e char_counts devem ter o mesmo comprimento.")
    if (weights <= 0).any():
        raise ValueError("char_counts devem ser positivos.")
    return values, weights


def character_weighted_mean(scores: Sequence[float], char_counts: Sequence[float]) -> float:
    """Return a finite score weighted by the number of characters."""

    values, weights = _weighted_inputs(scores, char_counts)
    result = float(np.dot(values, weights) / np.sum(weights))
    if not math.isfinite(result):
        raise ValueError("A média ponderada resultou não finita.")
    return result


def trimmed_character_weighted_mean(
    scores: Sequence[float],
    char_counts: Sequence[float],
    *,
    proportion: float = TRIM_PROPORTION,
) -> float:
    """Trim a fixed number of low/high-score windows, then retain char weights.

    The proportion is a pre-registered window-count proportion.  floor is
    used at each tail; fewer than three windows therefore remain untrimmed.
    Ties are resolved by stable input order.  The retained windows continue to
    contribute in proportion to their character counts.
    """

    values, weights = _weighted_inputs(scores, char_counts)
    try:
        trim = float(proportion)
    except (TypeError, ValueError) as error:
        raise ValueError("proportion deve ser numérica.") from error
    if not math.isfinite(trim) or trim < 0 or trim >= 0.5:
        raise ValueError("proportion deve estar em [0, 0.5).")
    trim_count = int(math.floor(values.size * trim))
    if trim_count * 2 >= values.size:
        raise ValueError("Aparar removeria todas as janelas.")
    if trim_count:
        order = np.argsort(values, kind="stable")
        keep = order[trim_count: values.size - trim_count]
        values, weights = values[keep], weights[keep]
    return character_weighted_mean(values, weights)


def aggregate_member_windows(
    windows: pd.DataFrame,
    *,
    member_score_columns: Sequence[str] = EXPECTED_MEMBER_SCORE_COLUMNS,
) -> pd.DataFrame:
    """Aggregate every window into per-article member and robust scores.

    windowTrimmedMean is the arithmetic mean of the three independently
    trimmed, character-weighted member scores.  This keeps member sampling
    symmetric while making the trimming rule explicit and reproducible.
    """

    if not isinstance(windows, pd.DataFrame) or windows.empty:
        raise ValueError("É necessário um DataFrame de janelas não vazio.")
    id_col = "articleId" if "articleId" in windows.columns else "id" if "id" in windows.columns else None
    weight_col = "charCount" if "charCount" in windows.columns else "windowCharacters" if "windowCharacters" in windows.columns else None
    if id_col is None or weight_col is None:
        raise ValueError("Janelas requerem articleId/id e charCount/windowCharacters.")
    score_columns = tuple(member_score_columns)
    if len(score_columns) != 3 or len(set(score_columns)) != 3 or not set(score_columns).issubset(windows.columns):
        raise ValueError("São necessários os três scores score_42, score_43 e score_44.")
    if "windowIndex" in windows.columns and windows.duplicated([id_col, "windowIndex"]).any():
        raise ValueError("Janelas duplicadas para o mesmo artigo.")
    ids = windows[id_col].astype(str)
    weights = pd.to_numeric(windows[weight_col], errors="coerce").to_numpy(dtype=float)
    score_values = windows[list(score_columns)].to_numpy(dtype=float)
    if not np.isfinite(weights).all() or (weights <= 0).any() or not np.isfinite(score_values).all():
        raise ValueError("Pesos e scores de janela devem ser finitos; pesos positivos.")

    frame = windows.copy(deep=True)
    frame["_articleId"] = ids
    frame["_charCount"] = weights
    records: list[dict[str, Any]] = []
    for article_id, group in frame.groupby("_articleId", sort=False):
        member_scores = [
            character_weighted_mean(group[column].to_numpy(dtype=float), group["_charCount"].to_numpy(dtype=float))
            for column in score_columns
        ]
        trimmed_scores = [
            trimmed_character_weighted_mean(group[column].to_numpy(dtype=float), group["_charCount"].to_numpy(dtype=float))
            for column in score_columns
        ]
        records.append({
            "articleId": str(article_id),
            "score_42": member_scores[0],
            "score_43": member_scores[1],
            "score_44": member_scores[2],
            "windowMean": float(np.mean(member_scores)),
            "windowMemberMedian": float(np.median(member_scores)),
            "windowTrimmedMean": float(np.mean(trimmed_scores)),
            "memberStd": float(np.std(member_scores, ddof=0)),
            "memberMin": float(np.min(member_scores)),
            "memberMax": float(np.max(member_scores)),
            "windowCount": int(len(group)),
            "totalCharacters": int(np.sum(group["_charCount"].to_numpy(dtype=float))),
        })
    result = pd.DataFrame.from_records(records)
    numeric = result[[*score_columns, "windowMean", "windowMemberMedian", "windowTrimmedMean", "memberStd"]].to_numpy(dtype=float)
    if not np.isfinite(numeric).all():
        raise ValueError("Scores agregados devem ser finitos.")
    return result


def validate_true_training_selection(selection: pd.DataFrame, *, expected_articles: int) -> bool:
    """Reject Fake or duplicate article rows in a one-window-per-True fit set."""

    if not isinstance(selection, pd.DataFrame) or selection.empty:
        raise ValueError("A seleção de treino não pode ser vazia.")
    id_col = "articleId" if "articleId" in selection.columns else "id" if "id" in selection.columns else None
    if id_col is None or "label" not in selection.columns:
        raise ValueError("A seleção requer articleId/id e label.")
    if not selection["label"].eq(0).all():
        raise ValueError("Fake não pode entrar na seleção/fit.")
    ids = selection[id_col].astype(str)
    if selection[id_col].isna().any() or ids.duplicated().any():
        raise ValueError("A seleção deve ter um único window por articleId.")
    if len(selection) != int(expected_articles):
        raise ValueError(f"A seleção deveria conter {expected_articles} artigos True.")
    return True


def _binary_metrics(labels: np.ndarray, scores: np.ndarray, cutoff: float) -> dict[str, float | None]:
    decisions = scores >= cutoff
    true_mask = labels == 0
    fake_mask = labels == 1
    tp = int(np.sum(fake_mask & decisions))
    fp = int(np.sum(true_mask & decisions))
    tn = int(np.sum(true_mask & ~decisions))
    fn = int(np.sum(fake_mask & ~decisions))
    denominator = {
        "accuracy": labels.size,
        "recall": tp + fn,
        "fpr": fp + tn,
    }
    return {
        "rocAuc": float(roc_auc_score(labels, scores)) if np.unique(labels).size == 2 else None,
        "averagePrecision": float(average_precision_score(labels, scores)) if np.unique(labels).size == 2 else None,
        "accuracy": float((tp + tn) / denominator["accuracy"]) if denominator["accuracy"] else None,
        "recall": float(tp / denominator["recall"]) if denominator["recall"] else None,
        "fpr": float(fp / denominator["fpr"]) if denominator["fpr"] else None,
    }


def paired_bootstrap(
    labels: Sequence[int],
    scores_by_case: Mapping[str, Sequence[float]],
    cutoffs: Mapping[str, float],
    *,
    baseline_case: str = "ensemble_q95",
    n_resamples: int = 1000,
    seed: int = 20260916,
) -> dict[str, Any]:
    """Calculate paired-by-article 95% bootstrap intervals.

    Each resample index vector is shared by every case, so differences in
    accuracy/recall/FPR use the same articles.  Resamples lacking a class are
    retained for threshold metrics and omitted only from AUC/AP quantiles.
    """

    target = np.asarray(labels, dtype=int)
    if target.ndim != 1 or target.size == 0 or not np.isin(target, [0, 1]).all() or np.unique(target).size < 2:
        raise ValueError("labels devem conter as duas classes 0=True e 1=Fake.")
    if baseline_case not in scores_by_case or baseline_case not in cutoffs:
        raise ValueError("baseline_case precisa existir nos scores e cortes.")
    if int(n_resamples) < 2:
        raise ValueError("n_resamples deve ser >= 2.")
    prepared: dict[str, np.ndarray] = {}
    for case, values in scores_by_case.items():
        vector = _vector(values, f"scores[{case}]")
        if vector.size != target.size:
            raise ValueError("Todos os scores devem ter o tamanho dos labels.")
        cutoff = float(cutoffs[case])
        if not math.isfinite(cutoff):
            raise ValueError(f"Corte não finito para {case}.")
        prepared[case] = vector
    rng = np.random.default_rng(int(seed))
    indices = rng.integers(0, target.size, size=(int(n_resamples), target.size))
    metric_names = ("rocAuc", "averagePrecision", "accuracy", "recall", "fpr")
    samples: dict[str, dict[str, np.ndarray]] = {
        case: {metric: np.full(int(n_resamples), np.nan, dtype=float) for metric in metric_names}
        for case in prepared
    }
    differences: dict[str, dict[str, np.ndarray]] = {
        case: {metric: np.full(int(n_resamples), np.nan, dtype=float) for metric in ("accuracy", "recall", "fpr")}
        for case in prepared
    }
    for index, draw in enumerate(indices):
        draw_labels = target[draw]
        for case, values in prepared.items():
            metric = _binary_metrics(draw_labels, values[draw], float(cutoffs[case]))
            for name in metric_names:
                if metric[name] is not None:
                    samples[case][name][index] = metric[name]
        baseline_metrics = {name: samples[baseline_case][name][index] for name in ("accuracy", "recall", "fpr")}
        for case in prepared:
            for name in ("accuracy", "recall", "fpr"):
                value = samples[case][name][index]
                if math.isfinite(value) and math.isfinite(baseline_metrics[name]):
                    differences[case][name][index] = value - baseline_metrics[name]

    def interval(values: np.ndarray) -> dict[str, Any]:
        finite = values[np.isfinite(values)]
        if finite.size == 0:
            return {"lower": None, "upper": None, "validResamples": 0}
        return {
            "lower": float(np.quantile(finite, 0.025)),
            "upper": float(np.quantile(finite, 0.975)),
            "validResamples": int(finite.size),
        }

    cases: dict[str, Any] = {}
    for case, values in prepared.items():
        point = _binary_metrics(target, values, float(cutoffs[case]))
        cases[case] = {
            "pointEstimate": point,
            "intervals95": {name: interval(samples[case][name]) for name in metric_names},
            "differenceVsBaseline": {
                name: interval(differences[case][name]) for name in ("accuracy", "recall", "fpr")
            },
        }
    return {
        "method": "paired_by_article_percentile_bootstrap",
        "seed": int(seed),
        "resamples": int(n_resamples),
        "confidenceLevel": 0.95,
        "baselineCase": baseline_case,
        "cases": cases,
    }


def selection_diversity(selections: Mapping[int, pd.DataFrame]) -> dict[str, Any]:
    """Summarize how many distinct training windows each article received."""

    if set(map(int, selections)) != {42, 43, 44}:
        raise ValueError("A diversidade requer exatamente as seeds 42, 43 e 44.")
    normalized: dict[int, pd.DataFrame] = {}
    for seed, frame in selections.items():
        id_col = "articleId" if "articleId" in frame.columns else "id" if "id" in frame.columns else None
        if id_col is None or "windowIndex" not in frame.columns:
            raise ValueError("Cada seleção requer ID e windowIndex.")
        validate_true_training_selection(frame, expected_articles=len(frame))
        copy = frame[[id_col, "windowIndex"]].copy()
        copy["articleId"] = copy[id_col].astype(str)
        normalized[int(seed)] = copy[["articleId", "windowIndex"]]
    ids = set(normalized[42]["articleId"])
    if any(set(frame["articleId"]) != ids for frame in normalized.values()):
        raise ValueError("As seleções não cobrem os mesmos IDs.")
    rows = []
    for article_id in normalized[42]["articleId"]:
        windows = [int(normalized[seed].set_index("articleId").loc[article_id, "windowIndex"]) for seed in (42, 43, 44)]
        rows.append({"articleId": article_id, "distinctWindowCount": len(set(windows)), "windowIndices": windows})
    counts = pd.Series([row["distinctWindowCount"] for row in rows]).value_counts().sort_index()
    pairwise = {}
    for left, right in combinations((42, 43, 44), 2):
        left_map = normalized[left].set_index("articleId")["windowIndex"]
        right_map = normalized[right].set_index("articleId")["windowIndex"]
        pairwise[f"{left}_{right}"] = float(np.mean(left_map.to_numpy() == right_map.reindex(left_map.index).to_numpy()))
    return {
        "articleCount": len(rows),
        "distribution": {str(int(key)): int(value) for key, value in counts.items()},
        "pairwiseSameWindowRate": pairwise,
        "perArticle": rows,
    }


def fuse_rank_scores(
    full_reference: Sequence[float],
    window_reference: Sequence[float],
    full_values: Sequence[float],
    window_values: Sequence[float],
    *,
    weights: tuple[float, float] = (0.5, 0.5),
) -> np.ndarray:
    """Fuse full-text and window ensemble ECDF ranks with frozen 50/50 weights."""

    if tuple(weights) != (0.5, 0.5):
        raise ValueError("hybrid_rank_q95 exige pesos fixos (0.5, 0.5).")
    full = ecdf_percentile(full_reference, full_values)
    window = ecdf_percentile(window_reference, window_values)
    if full.size != window.size:
        raise ValueError("As duas vistas devem ter o mesmo número de artigos.")
    result = 0.5 * full + 0.5 * window
    if not np.isfinite(result).all():
        raise ValueError("hybridScore resultou não finito.")
    return result


__all__ = [
    "EXPECTED_MEMBER_SCORE_COLUMNS",
    "TRIM_PROPORTION",
    "aggregate_member_windows",
    "align_by_id",
    "character_weighted_mean",
    "ecdf_percentile",
    "fuse_rank_scores",
    "paired_bootstrap",
    "selection_diversity",
    "trimmed_character_weighted_mean",
    "validate_true_training_selection",
]


In [ ]:
"""Runtime cell source embedded in the standalone hybrid notebook.

The builder injects this file as a notebook cell.  It intentionally relies
only on variables created by the preceding self-contained notebook cells and
does not import project modules at runtime.
"""

import json
import math
import platform
import time
from importlib.metadata import version as package_version
from urllib.parse import urlparse

from sklearn.base import clone
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)


HYBRID_STARTED = time.perf_counter()
HYBRID_CONFIG = {
    "corpusRevision": CORPUS_REVISION,
    "windowSize": WINDOW_SIZE,
    "featureOrder": list(anomalyColumns),
    "samplingSeeds": [42, 43, 44],
    "estimatorSeed": 42,
    "nEstimators": 300,
    "quantiles": [0.95, 0.97],
    "rankFusion": {"weights": {"fullText": 0.5, "windowEnsemble": 0.5}, "ecdf": "P(score <= x)"},
    "trimmedMean": {"proportion": TRIM_PROPORTION, "rule": "floor window-count at each score tail; retain character weights"},
    "positiveLabel": 1,
    "aggregation": "character_weighted_mean_then_equal_member_mean",
    "exploratoryTest": True,
}


def _linear_quantile(values, quantile):
    values = np.asarray(values, dtype=float)
    if values.ndim != 1 or values.size == 0 or not np.isfinite(values).all():
        raise ValueError("Valores para quantil devem ser um vetor finito não vazio.")
    try:
        return float(np.quantile(values, quantile, method="linear"))
    except TypeError:
        return float(np.quantile(values, quantile, interpolation="linear"))


def _finite_or_none(value):
    if value is None:
        return None
    try:
        number = float(value)
    except (TypeError, ValueError):
        return value
    return number if np.isfinite(number) else None


def _json_safe(value):
    if isinstance(value, dict):
        return {str(key): _json_safe(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [_json_safe(item) for item in value]
    if isinstance(value, np.generic):
        return _json_safe(value.item())
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    if value is pd.NA or value is None:
        return None
    if isinstance(value, float):
        return value if np.isfinite(value) else None
    try:
        if pd.isna(value):
            return None
    except (TypeError, ValueError):
        pass
    return value


def _score_pipeline(pipeline, frame):
    scores = -np.asarray(pipeline.decision_function(frame[anomalyColumns]), dtype=float)
    if scores.ndim != 1 or not np.isfinite(scores).all():
        raise ValueError("Scores do modelo devem ser finitos.")
    return scores


def _source_domain(value):
    if value is None or (isinstance(value, float) and not np.isfinite(value)):
        return None
    text = str(value).strip()
    if not text:
        return None
    return urlparse(text).hostname


def _metric_row(labels, scores, cutoff):
    labels = np.asarray(labels, dtype=int)
    scores = np.asarray(scores, dtype=float)
    decisions = scores >= float(cutoff)
    tn, fp, fn, tp = confusion_matrix(labels, decisions.astype(int), labels=[0, 1]).ravel()
    total = int(labels.size)
    def ratio(numerator, denominator):
        return None if denominator == 0 else float(numerator / denominator)
    auc = None if np.unique(labels).size < 2 else float(roc_auc_score(labels, scores))
    ap = None if np.unique(labels).size < 2 else float(average_precision_score(labels, scores))
    return {
        "TP": int(tp), "FP": int(fp), "TN": int(tn), "FN": int(fn), "total": total,
        "fakeDetected": int(tp), "fakeTotal": int(np.sum(labels == 1)), "fakeMissed": int(fn),
        "trueSignaled": int(fp), "trueTotal": int(np.sum(labels == 0)), "trueWithoutAlert": int(tn),
        "correct": int(tp + tn), "accuracy": ratio(tp + tn, total),
        "precision": ratio(tp, tp + fp), "recall": ratio(tp, tp + fn),
        "f1": ratio(2 * tp, 2 * tp + fp + fn), "fpr": ratio(fp, fp + tn),
        "rocAuc": auc, "averagePrecision": ap, "ROC-AUC": auc, "AP": ap,
        "decisionCount": int(decisions.sum()),
    }


def _metric_delta(row, baseline):
    fields = ("TP", "FP", "TN", "FN", "accuracy", "precision", "recall", "f1", "fpr", "rocAuc", "averagePrecision")
    delta = {}
    for field in fields:
        left, right = row.get(field), baseline.get(field)
        delta[field] = None if left is None or right is None else float(left - right)
    fp_avoided = baseline["FP"] - row["FP"]
    fn_additional = row["FN"] - baseline["FN"]
    delta.update({
        "fpAvoided": int(fp_avoided),
        "fnAdditional": int(fn_additional),
        "netFpAvoidedMinusFnAdditional": int(fp_avoided - fn_additional),
    })
    return delta


def _aggregate_rows_for_partition(partition_name, windows):
    frame = windows.copy(deep=True)
    frame["articleId"] = frame["id"].astype(str)
    aggregated = aggregate_member_windows(frame)
    aggregated["partition"] = partition_name
    return aggregated


def _validate_partition_contract(partitions, all_article_ids):
    expected = {
        "normalTrain": 0,
        "normalValidation": 0,
        "normalTest": 0,
        "fakeValidation": 1,
        "fakeTest": 1,
    }
    seen = set()
    if set(partitions) != set(expected):
        raise ValueError("Partições incompletas ou inesperadas.")
    for name, expected_label in expected.items():
        frame = partitions[name]
        if frame.empty or not frame["id"].is_unique or not frame["label"].eq(expected_label).all():
            raise ValueError(f"Partição inválida: {name}.")
        ids = set(frame["id"].astype(str))
        if seen & ids:
            raise ValueError("IDs atravessam partições.")
        seen |= ids
    if seen != set(map(str, all_article_ids)):
        raise ValueError("Partições não cobrem exatamente o corpus.")
    return True


partition_frames = {
    "normalTrain": normalTrainFrame,
    "normalValidation": normalValidationFrame,
    "normalTest": normalTestFrame,
    "fakeValidation": fakeValidationFrame,
    "fakeTest": fakeTestFrame,
}

for _name, _frame in partition_frames.items():
    if not _frame["id"].is_unique:
        raise ValueError(f"IDs duplicados em {_name}.")
_validate_partition_contract(partition_frames, featuresFrame["id"].astype(str).tolist())


# Fit the three window members independently.  The full-text and prefix
# pipelines were already fitted in the preceding notebook cells using only
# normalTrainFrame; this loop keeps the member 42 protocol identical.
member_pipelines = {}
member_selections = {}
_fit_started = time.perf_counter()
for _sampling_seed in (42, 43, 44):
    _selected = selectTrainingWindows(windowPartitions["normalTrain"], seed=_sampling_seed)
    validate_true_training_selection(_selected, expected_articles=len(normalTrainFrame))
    _member_pipeline = clone(anomalyPipeline)
    fitNormalOnly(_member_pipeline, _selected)
    member_pipelines[_sampling_seed] = _member_pipeline
    member_selections[_sampling_seed] = _selected.copy(deep=True)
fit_seconds = float(time.perf_counter() - _fit_started)


_window_scoring_started = time.perf_counter()
window_score_frames = []
for _partition_name, _window_frame in windowPartitions.items():
    _base = _window_frame[["id", "windowIndex", "windowCharacters", *anomalyColumns]].copy()
    _base["id"] = _base["id"].astype(str)
    _base["charStart"] = _base.groupby("id", sort=False)["windowCharacters"].cumsum() - _base["windowCharacters"]
    _base["charEnd"] = _base["charStart"] + _base["windowCharacters"]
    for _seed, _pipeline in member_pipelines.items():
        _base[f"score_{_seed}"] = _score_pipeline(_pipeline, _window_frame)
    _base["windowMean"] = _base[["score_42", "score_43", "score_44"]].mean(axis=1)
    _base["windowMemberMedian"] = _base[["score_42", "score_43", "score_44"]].median(axis=1)
    _base["memberStd"] = _base[["score_42", "score_43", "score_44"]].std(axis=1, ddof=0)
    _base["partition"] = _partition_name
    window_score_frames.append(_base)
window_scores_frame = pd.concat(window_score_frames, ignore_index=True)
window_scoring_seconds = float(time.perf_counter() - _window_scoring_started)
if not np.isfinite(window_scores_frame[["score_42", "score_43", "score_44", "windowMean", "windowMemberMedian", "memberStd"]].to_numpy(dtype=float)).all():
    raise ValueError("Há score de janela ausente ou não finito.")


_article_scoring_started = time.perf_counter()
article_score_frames = []
features_by_id = featuresFrame.set_index("id", drop=False)
for _partition_name in ("normalValidation", "fakeValidation", "normalTest", "fakeTest"):
    _partition = partition_frames[_partition_name]
    _partition_windows = window_scores_frame.loc[window_scores_frame["partition"].eq(_partition_name)].copy()
    _aggregated = _aggregate_rows_for_partition(_partition_name, _partition_windows)
    _ids = _aggregated["articleId"].tolist()
    _full_features = fullFeaturesFrame.loc[_ids]
    _prefix_features = features_by_id.loc[_ids]
    _aggregated["fullScore"] = _score_pipeline(fullPipeline, _full_features)
    _aggregated["prefixScore"] = _score_pipeline(anomalyPipeline, _prefix_features)
    _meta = newsFrame.set_index("id").loc[_ids]
    _aggregated["label"] = _partition.set_index("id").loc[_ids, "label"].to_numpy(dtype=int)
    _aggregated["authorPresent"] = _meta["tem_autor"].to_numpy()
    _aggregated["category"] = _meta["categoria"].to_numpy()
    _aggregated["sourceDomain"] = _meta["link"].map(_source_domain).to_numpy()
    _aggregated["originalWordCount"] = pd.to_numeric(_meta["num_palavras"], errors="coerce").to_numpy()
    article_score_frames.append(_aggregated)
article_scores_frame = pd.concat(article_score_frames, ignore_index=True)
article_scoring_seconds = float(time.perf_counter() - _article_scoring_started)

_validation_scores = article_scores_frame.loc[article_scores_frame["partition"].eq("normalValidation")].copy()
_full_reference = _validation_scores["fullScore"].to_numpy(dtype=float)
_window_reference = _validation_scores["windowMean"].to_numpy(dtype=float)
if len(_validation_scores) != 720:
    raise ValueError("A calibração híbrida requer as 720 True de validação.")
article_scores_frame["fullTextPercentile"] = ecdf_percentile(
    _full_reference, article_scores_frame["fullScore"].to_numpy(dtype=float)
)
article_scores_frame["windowEnsemblePercentile"] = ecdf_percentile(
    _window_reference, article_scores_frame["windowMean"].to_numpy(dtype=float)
)
article_scores_frame["hybridScore"] = (
    0.5 * article_scores_frame["fullTextPercentile"]
    + 0.5 * article_scores_frame["windowEnsemblePercentile"]
)
if not np.isfinite(article_scores_frame["hybridScore"].to_numpy(dtype=float)).all():
    raise ValueError("hybridScore ausente ou não finito.")
_validation_scores = article_scores_frame.loc[article_scores_frame["partition"].eq("normalValidation")].copy()


score_columns = {
    "full_text": "fullScore",
    "prefix": "prefixScore",
    "single": "score_42",
    "ensemble": "windowMean",
    "median": "windowMemberMedian",
    "trimmed": "windowTrimmedMean",
    "hybrid": "hybridScore",
}
validation_thresholds = {}
for _key, _column in score_columns.items():
    validation_values = _validation_scores[_column].to_numpy(dtype=float)
    validation_thresholds[_key] = {
        str(_quantile): {
            "cutoff": _linear_quantile(validation_values, _quantile),
            "validationFPR": float(np.mean(validation_values >= _linear_quantile(validation_values, _quantile))),
            "validationCount": int(len(validation_values)),
        }
        for _quantile in (0.95, 0.97)
    }

scenario_specs = [
    ("full_text_q95", "full_text", 0.95),
    ("prefix_q95", "prefix", 0.95),
    ("single_q95", "single", 0.95),
    ("ensemble_q95", "ensemble", 0.95),
    ("hybrid_rank_q95", "hybrid", 0.95),
    ("single_q97", "single", 0.97),
    ("ensemble_q97", "ensemble", 0.97),
    ("hybrid_rank_q97", "hybrid", 0.97),
    ("window_member_median_q95", "median", 0.95),
    ("window_trimmed_mean_q95", "trimmed", 0.95),
    ("full_text_q97", "full_text", 0.97),
    ("prefix_q97", "prefix", 0.97),
    ("window_member_median_q97", "median", 0.97),
    ("window_trimmed_mean_q97", "trimmed", 0.97),
]

for _scenario, _model, _quantile in scenario_specs:
    _column = score_columns[_model]
    _cutoff = validation_thresholds[_model][str(_quantile)]["cutoff"]
    article_scores_frame[f"decision_{_scenario}"] = article_scores_frame[_column] >= _cutoff

test_articles_frame = article_scores_frame.loc[
    article_scores_frame["partition"].isin(["normalTest", "fakeTest"])
].copy()
test_articles_frame = pd.concat([
    test_articles_frame.loc[test_articles_frame["partition"].eq("normalTest")],
    test_articles_frame.loc[test_articles_frame["partition"].eq("fakeTest")],
], ignore_index=True)
test_labels = test_articles_frame["label"].to_numpy(dtype=int)
if test_labels.tolist() != testFrame["label"].to_numpy(dtype=int).tolist():
    raise ValueError("A ordem dos IDs de teste não coincide com o protocolo.")


comparison_rows = []
scenario_decisions = {}
for _scenario, _model, _quantile in scenario_specs:
    _column = score_columns[_model]
    _scores = test_articles_frame[_column].to_numpy(dtype=float)
    _cutoff_data = validation_thresholds[_model][str(_quantile)]
    _cutoff = _cutoff_data["cutoff"]
    _row = _metric_row(test_labels, _scores, _cutoff)
    _row.update({
        "modelKey": _scenario,
        "model": _model,
        "quantile": _quantile,
        "cutoff": _cutoff,
        "threshold": _cutoff,
        "validationFPR": _cutoff_data["validationFPR"],
        "scoreColumn": _column,
        "decisionRule": "score >= cutoff",
    })
    comparison_rows.append(_row)
    scenario_decisions[_scenario] = _scores >= _cutoff

baseline_single = next(row for row in comparison_rows if row["modelKey"] == "single_q95")
baseline_ensemble = next(row for row in comparison_rows if row["modelKey"] == "ensemble_q95")
for _row in comparison_rows:
    _row["deltaVsSingleQ95"] = _metric_delta(_row, baseline_single)
    _row["deltaVsEnsembleQ95"] = _metric_delta(_row, baseline_ensemble)


comparison_transitions = []
for _index, _article_id in enumerate(test_articles_frame["articleId"].astype(str)):
    _baseline_decision = bool(scenario_decisions["single_q95"][_index])
    _decisions = {scenario: bool(values[_index]) for scenario, values in scenario_decisions.items()}
    _transition = {
        scenario: (
            "baseline" if scenario == "single_q95"
            else "mantido" if decision == _baseline_decision
            else "adicionado" if decision else "removido"
        )
        for scenario, decision in _decisions.items()
    }
    comparison_transitions.append({
        "articleId": str(_article_id),
        "label": int(test_labels[_index]),
        "decisions": _decisions,
        **_decisions,
        "transition": _transition,
        "transitionFromSingleQ95": _transition,
    })


def _window_details(article_id, *, order_column="windowMean"):
    _group = window_scores_frame.loc[window_scores_frame["id"].eq(str(article_id))].copy()
    _group = _group.sort_values([order_column, "windowIndex"], ascending=[False, True], kind="stable")
    _total = float(_group["windowCharacters"].sum())
    rows = []
    for _, _row in _group.iterrows():
        rows.append({
            "articleId": str(article_id),
            "windowIndex": int(_row["windowIndex"]),
            "charStart": int(_row["charStart"]),
            "charEnd": int(_row["charEnd"]),
            "charCount": int(_row["windowCharacters"]),
            "features": {feature: _finite_or_none(_row[feature]) for feature in anomalyColumns},
            "score_42": float(_row["score_42"]),
            "score_43": float(_row["score_43"]),
            "score_44": float(_row["score_44"]),
            "windowMean": float(_row["windowMean"]),
            "windowMemberMedian": float(_row["windowMemberMedian"]),
            "memberStd": float(_row["memberStd"]),
            "weightedContribution": float(_row["windowMean"] * _row["windowCharacters"] / _total),
        })
    return rows


def _article_diagnostic(article_row):
    _row = article_row
    _windows = _window_details(_row["articleId"])
    _sorted = sorted(_windows, key=lambda value: (value["windowMean"], -value["windowIndex"]))
    return {
        "articleId": str(_row["articleId"]),
        "partition": _row["partition"],
        "label": int(_row["label"]),
        "authorPresent": None if pd.isna(_row["authorPresent"]) else bool(_row["authorPresent"]),
        "category": None if pd.isna(_row["category"]) else str(_row["category"]),
        "sourceDomain": None if pd.isna(_row["sourceDomain"]) else str(_row["sourceDomain"]),
        "originalWordCount": _finite_or_none(_row["originalWordCount"]),
        "windowCount": int(_row["windowCount"]),
        "totalCharacters": int(_row["totalCharacters"]),
        "fullScore": float(_row["fullScore"]),
        "windowScore": float(_row["windowMean"]),
        "fullTextPercentile": float(_row["fullTextPercentile"]),
        "windowEnsemblePercentile": float(_row["windowEnsemblePercentile"]),
        "hybridScore": float(_row["hybridScore"]),
        "singleScoreBySeed": {str(seed): float(_row[f"score_{seed}"]) for seed in (42, 43, 44)},
        "memberStd": float(_row["memberStd"]),
        "decisions": {scenario: bool(_row[f"decision_{scenario}"]) for scenario, _, _ in scenario_specs},
        "topWindows": _windows[:3],
        "leastAnomalousWindows": _sorted[:3],
    }


test_article_records = [_article_diagnostic(row) for _, row in test_articles_frame.iterrows()]
validation_article_frame = article_scores_frame.loc[
    article_scores_frame["partition"].eq("normalValidation")
]
fake_validation_article_frame = article_scores_frame.loc[
    article_scores_frame["partition"].eq("fakeValidation")
]
evaluation_article_records = [
    _article_diagnostic(row)
    for _, row in pd.concat([
        validation_article_frame,
        fake_validation_article_frame,
        test_articles_frame,
    ], ignore_index=True).iterrows()
]


def _error_ids(frame, scenario):
    return set(frame.loc[(frame["label"] == 1) & ~frame[f"decision_{scenario}"], "articleId"].astype(str))


def _false_positive_ids(frame, scenario):
    return set(frame.loc[(frame["label"] == 0) & frame[f"decision_{scenario}"], "articleId"].astype(str))


single_fn = _error_ids(test_articles_frame, "single_q95")
ensemble_fn = _error_ids(test_articles_frame, "ensemble_q95")
hybrid_fn = _error_ids(test_articles_frame, "hybrid_rank_q95")
single_fp = _false_positive_ids(test_articles_frame, "single_q95")
ensemble_fp = _false_positive_ids(test_articles_frame, "ensemble_q95")
hybrid_fp = _false_positive_ids(test_articles_frame, "hybrid_rank_q95")

false_negative_analysis = {
    "singleQ95Ids": sorted(single_fn),
    "ensembleQ95Ids": sorted(ensemble_fn),
    "hybridRankQ95Ids": sorted(hybrid_fn),
    "recoveredByHybrid": sorted(single_fn - hybrid_fn),
    "addedByHybrid": sorted(hybrid_fn - single_fn),
    "sameAsEnsemble": hybrid_fn == ensemble_fn,
    "details": [
        _article_diagnostic(row)
        for _, row in test_articles_frame.loc[
            test_articles_frame["articleId"].isin(sorted(single_fn | hybrid_fn))
        ].sort_values("articleId").iterrows()
    ],
}

false_positive_analysis = {
    "singleQ95Ids": sorted(single_fp),
    "ensembleQ95Ids": sorted(ensemble_fp),
    "hybridRankQ95Ids": sorted(hybrid_fp),
    "removedByEnsemble": sorted(single_fp - ensemble_fp),
    "removedByHybridFromSingle": sorted(single_fp - hybrid_fp),
    "addedByHybridFromSingle": sorted(hybrid_fp - single_fp),
    "ensembleKeepsSingle": sorted(ensemble_fp & single_fp),
    "focus1814t": next(
        (_article_diagnostic(row) for _, row in test_articles_frame.loc[
            test_articles_frame["articleId"].eq("1814t")
        ].iterrows()),
        None,
    ),
    "details": [
        _article_diagnostic(row)
        for _, row in test_articles_frame.loc[
            test_articles_frame["articleId"].isin(sorted(single_fp | ensemble_fp | hybrid_fp))
        ].sort_values("articleId").iterrows()
    ],
}


full_alert = test_articles_frame["decision_full_text_q95"].to_numpy(dtype=bool)
window_alert = test_articles_frame["decision_ensemble_q95"].to_numpy(dtype=bool)
full_fn = _error_ids(test_articles_frame, "full_text_q95")
window_fn = ensemble_fn
full_fp = _false_positive_ids(test_articles_frame, "full_text_q95")
window_fp = ensemble_fp
full_set = set(test_articles_frame.loc[full_alert, "articleId"].astype(str))
window_set = set(test_articles_frame.loc[window_alert, "articleId"].astype(str))
fake_full_alert_ids = set(test_articles_frame.loc[(test_articles_frame["label"] == 1) & test_articles_frame["decision_full_text_q95"], "articleId"].astype(str))
fake_window_alert_ids = set(test_articles_frame.loc[(test_articles_frame["label"] == 1) & test_articles_frame["decision_ensemble_q95"], "articleId"].astype(str))
complementarity = {
    "falsePositives": {
        "common": sorted(full_fp & window_fp),
        "fullTextExclusive": sorted(full_fp - window_fp),
        "windowExclusive": sorted(window_fp - full_fp),
    },
    "falseNegatives": {
        "common": sorted(full_fn & window_fn),
        "fullTextExclusive": sorted(full_fn - window_fn),
        "windowExclusive": sorted(window_fn - full_fn),
    },
    "fakeRecoveredByFullLostByWindow": sorted(fake_full_alert_ids - fake_window_alert_ids),
    "fakeRecoveredByWindowLostByFull": sorted(fake_window_alert_ids - fake_full_alert_ids),
    "decisionTable2x2": {
        "fullNoWindowNo": int(np.sum(~full_alert & ~window_alert)),
        "fullNoWindowYes": int(np.sum(~full_alert & window_alert)),
        "fullYesWindowNo": int(np.sum(full_alert & ~window_alert)),
        "fullYesWindowYes": int(np.sum(full_alert & window_alert)),
    },
    "spearmanFullVsWindow": float(test_articles_frame["fullScore"].corr(test_articles_frame["windowMean"], method="spearman")),
    "spearmanFullVsHybridWindowPercentile": float(test_articles_frame["fullTextPercentile"].corr(test_articles_frame["windowEnsemblePercentile"], method="spearman")),
    "largestRankDisagreements": [
        {
            "articleId": str(row["articleId"]),
            "label": int(row["label"]),
            "fullTextPercentile": float(row["fullTextPercentile"]),
            "windowEnsemblePercentile": float(row["windowEnsemblePercentile"]),
            "hybridScore": float(row["hybridScore"]),
            "absolutePercentileGap": float(abs(row["fullTextPercentile"] - row["windowEnsemblePercentile"])),
        }
        for _, row in test_articles_frame.assign(
            absolutePercentileGap=lambda frame: (frame["fullTextPercentile"] - frame["windowEnsemblePercentile"]).abs()
        ).nlargest(20, "absolutePercentileGap").iterrows()
    ],
    "contributionNote": "Percentis 50/50 descrevem a regra aritmética da decisão; não são explicação causal da falsidade.",
}


member_correlation = {}
for left, right in ((42, 43), (42, 44), (43, 44)):
    member_correlation[f"{left}_{right}"] = float(
        test_articles_frame[f"score_{left}"].corr(test_articles_frame[f"score_{right}"], method="spearman")
    )
member_std = test_articles_frame["memberStd"].to_numpy(dtype=float)
member_comparisons = []
for _seed in (42, 43, 44):
    _column = f"score_{_seed}"
    _validation_values = _validation_scores[_column].to_numpy(dtype=float)
    _seed_thresholds = {
        str(_quantile): {
            "cutoff": _linear_quantile(_validation_values, _quantile),
            "validationFPR": float(np.mean(_validation_values >= _linear_quantile(_validation_values, _quantile))),
        }
        for _quantile in (0.95, 0.97)
    }
    _seed_row = {"seed": _seed, "thresholds": _seed_thresholds, "comparisons": {}}
    for _quantile in (0.95, 0.97):
        _cutoff = _seed_thresholds[str(_quantile)]["cutoff"]
        _seed_row["comparisons"][f"q{int(_quantile * 100)}"] = _metric_row(
            test_labels, test_articles_frame[_column].to_numpy(dtype=float), _cutoff
        )
    member_comparisons.append(_seed_row)

selection_diversity_result = selection_diversity(member_selections)
stability = {
    "spearmanBetweenMembers": member_correlation,
    "validationSpearmanBetweenMembers": {
        f"{left}_{right}": float(_validation_scores[f"score_{left}"].corr(_validation_scores[f"score_{right}"], method="spearman"))
        for left, right in ((42, 43), (42, 44), (43, 44))
    },
    "memberStdByArticle": {
        "mean": float(np.mean(member_std)),
        "median": float(np.median(member_std)),
        "max": float(np.max(member_std)),
        "q95": float(np.quantile(member_std, 0.95)),
    },
    "decisionChangesBetweenSeeds": {
        f"{left}_{right}": int(np.sum(
            (test_articles_frame[f"score_{left}"].to_numpy(dtype=float) >= member_comparisons[left - 42]["thresholds"]["0.95"]["cutoff"])
            != (test_articles_frame[f"score_{right}"].to_numpy(dtype=float) >= member_comparisons[right - 42]["thresholds"]["0.95"]["cutoff"])
        ))
        for left, right in ((42, 43), (42, 44), (43, 44))
    },
    "individualMembers": member_comparisons,
    "selectionDiversity": selection_diversity_result,
}


def _subgroup_metrics(frame, scenario, dimension):
    rows = []
    scenario_lookup = {spec[0]: spec[1] for spec in scenario_specs}
    for value, group in frame.groupby(dimension, dropna=False, sort=False):
        label = None if pd.isna(value) else value
        labels = group["label"].to_numpy(dtype=int)
        scores = group[score_columns[scenario_lookup[scenario]]].to_numpy(dtype=float)
        cutoff = next(row["cutoff"] for row in comparison_rows if row["modelKey"] == scenario)
        metric = _metric_row(labels, scores, cutoff)
        metric.update({"dimension": dimension, "group": _json_safe(label), "denominator": int(len(group)), "scenario": scenario})
        rows.append(metric)
    return rows


test_articles_frame["lengthQuartile"] = pd.qcut(
    pd.to_numeric(test_articles_frame["originalWordCount"], errors="coerce"),
    q=4,
    labels=["Q1", "Q2", "Q3", "Q4"],
    duplicates="drop",
)
stratified_metrics = {
    dimension: _subgroup_metrics(test_articles_frame, "single_q95", dimension)
    for dimension in ("authorPresent", "category", "sourceDomain", "lengthQuartile")
}
stratified_metrics["hybrid_rank_q95"] = {
    dimension: _subgroup_metrics(test_articles_frame, "hybrid_rank_q95", dimension)
    for dimension in ("authorPresent", "category", "sourceDomain", "lengthQuartile")
}


_bootstrap_started = time.perf_counter()
test_score_map = {
    row["modelKey"]: test_articles_frame[score_columns[row["model"]]].to_numpy(dtype=float)
    for row in comparison_rows
}
test_cutoff_map = {row["modelKey"]: row["cutoff"] for row in comparison_rows}
bootstrap_result = paired_bootstrap(
    test_labels,
    test_score_map,
    test_cutoff_map,
    baseline_case="ensemble_q95",
    n_resamples=1000,
    seed=20260916,
)
bootstrap_seconds = float(time.perf_counter() - _bootstrap_started)


article_score_columns = [
    "articleId", "partition", "label", "fullScore", "prefixScore",
    "score_42", "score_43", "score_44", "windowMean", "windowMemberMedian",
    "windowTrimmedMean", "memberStd", "fullTextPercentile",
    "windowEnsemblePercentile", "hybridScore", "windowCount", "totalCharacters",
    "authorPresent", "category", "sourceDomain", "originalWordCount",
]
article_score_export = []
for _, _row in article_scores_frame.iterrows():
    _record = {column: _json_safe(_row[column]) for column in article_score_columns}
    _record["thresholds"] = {
        scenario: float(next(row["cutoff"] for row in comparison_rows if row["modelKey"] == scenario))
        for scenario, _, _ in scenario_specs
    }
    _record["decisions"] = {
        scenario: bool(_row[f"decision_{scenario}"]) for scenario, _, _ in scenario_specs
    }
    for scenario, _, _ in scenario_specs:
        _record[f"decision_{scenario}"] = _record["decisions"][scenario]
    article_score_export.append(_record)

window_score_export = [
    {
        "articleId": str(_row["id"]),
        "partition": str(_row["partition"]),
        "windowIndex": int(_row["windowIndex"]),
        "charStart": int(_row["charStart"]),
        "charEnd": int(_row["charEnd"]),
        "charCount": int(_row["windowCharacters"]),
        **{_feature: _finite_or_none(_row[_feature]) for _feature in anomalyColumns},
        **{_column: float(_row[_column]) for _column in ("score_42", "score_43", "score_44", "windowMean", "windowMemberMedian", "memberStd")},
    }
    for _, _row in window_scores_frame.iterrows()
]

training_selection_export = []
for _seed, _selection in member_selections.items():
    _selection_frame = _selection[["id", "windowIndex", "windowCharacters"]].copy()
    _offsets = window_scores_frame.loc[
        window_scores_frame["partition"].eq("normalTrain"),
        ["id", "windowIndex", "charStart", "charEnd"],
    ]
    _selection_frame = _selection_frame.merge(_offsets, on=["id", "windowIndex"], how="left", validate="one_to_one")
    for _, _row in _selection_frame.iterrows():
        training_selection_export.append({
            "articleId": str(_row["id"]),
            "memberSeed": int(_seed),
            "windowIndex": int(_row["windowIndex"]),
            "charStart": int(_row["charStart"]),
            "charEnd": int(_row["charEnd"]),
            "charCount": int(_row["windowCharacters"]),
        })


partition_ids_export = {
    name: [str(value) for value in frame["id"].tolist()]
    for name, frame in partition_frames.items()
}
partition_counts_export = {
    name: {
        "articles": int(frame["id"].nunique()),
        "label": int(frame["label"].iloc[0]),
        "windows": int(window_scores_frame.loc[window_scores_frame["partition"].eq(name), "id"].size),
    }
    for name, frame in partition_frames.items()
}

required_baseline_row = next(row for row in comparison_rows if row["modelKey"] == "single_q95")
if (required_baseline_row["TP"], required_baseline_row["FP"], required_baseline_row["TN"], required_baseline_row["FN"]) != (1771, 22, 698, 29):
    raise ValueError("O membro seed 42 não reproduziu o single_q95 esperado.")

evaluation_report = {
    "comparisons": comparison_rows,
    "transitions": comparison_transitions,
    "ordering": [
        {
            "modelKey": row["modelKey"],
            "scoreColumn": row["scoreColumn"],
            "rocAuc": row["rocAuc"],
            "averagePrecision": row["averagePrecision"],
            "note": "ROC-AUC/AP medem ordenação; o threshold governa decisões.",
        }
        for row in comparison_rows
    ],
    "articleScores": article_score_export,
    "windowScores": window_score_export,
    "trainingSelections": training_selection_export,
    "thresholds": validation_thresholds,
    "partitionIds": partition_ids_export,
    "partitionCounts": partition_counts_export,
    "config": HYBRID_CONFIG,
    "diagnostics": {
        "falseNegatives": false_negative_analysis,
        "falsePositives": false_positive_analysis,
        "complementarity": complementarity,
        "stability": stability,
        "stratifiedMetrics": stratified_metrics,
    },
    "bootstrap": bootstrap_result,
    "status": "exploratory_test_inspected",
    "warnings": [
        "O teste histórico já foi inspecionado; os resultados são avaliação interna exploratória.",
        "Anomaly score não é probabilidade de falsidade e não confirma veracidade factual.",
        "Nenhum Fake entrou no fit ou na calibração q95/q97; Fake validation foi usada somente para avaliação.",
        "Pesos híbridos 0.5/0.5, seeds, features, agregadores e regra de corte foram congelados antes do teste.",
        "A média aparada usa proporção pré-registrada de 10% por cauda e mantém ponderação por caracteres nos trechos retidos.",
        "Não houve seleção automática pelo teste; intervalos compatíveis com empate não demonstram superioridade.",
        "Colab não foi testado; esta execução é local em kernel isolado.",
    ],
    "timings": {
        "hybridRuntimeSeconds": float(time.perf_counter() - HYBRID_STARTED),
        "fitSeconds": fit_seconds,
        "windowScoringSeconds": window_scoring_seconds,
        "articleScoringSeconds": article_scoring_seconds,
        "bootstrapSeconds": bootstrap_seconds,
    },
}

hybrid_export = {
    "evaluationReport": evaluation_report,
    "diagnostic": {
        "falseNegatives": false_negative_analysis,
        "falsePositives": false_positive_analysis,
        "complementarity": complementarity,
        "stability": stability,
        "stratifiedMetrics": stratified_metrics,
    },
    "articleScores": article_score_export,
    "windowScores": window_score_export,
    "trainingSelections": training_selection_export,
    "partitionIds": partition_ids_export,
    "partitionCounts": partition_counts_export,
    "config": HYBRID_CONFIG,
    "bootstrap": bootstrap_result,
}


## 26. Saída para o executor

A variável `hybrid_export` é serializada pelo executor em um novo diretório de artefatos. Inclui scores por artigo/janela, seleções, diagnósticos, comparações e bootstrap pareado.